# Experiment 87: V4.5 피처 — 정밀 정제

**기반**: Exp 85 V4 (LB **10.0480** — 현재 PB), 219 피처

**목표**: LB 10.0480 → 10.035~10.045

## 핵심 발견

V4 importance 진단에서 발견:
- V4 신규 42개 중 **단 3개만 effective** (M 그룹 상위 3개)
- N, O, P 그룹은 모두 noise

V5 시도에서 검증:
- 다양한 차원 percentile 확장 → V4 못 이김 (+0.0019)
- → **차원이 아니라 specific한 형태(scen_pct)가 핵심**

## 노트북 구조

이 노트북은 Exp 79 + Exp 82 + Exp 84 + Exp 85 + Exp 86 + Exp 87을 모두 포함합니다.

### 시나리오 A — 메모리 살아있는 경우 (권장)
Exp 86 V5 1-seed 실행 후라면 메모리에:
- `result_v4`, `features_v4`, `avg_rank_v4` (V4 학습 + 진단)
- `result_v3`, `result_v2`, `result_r2`
- `train_combined_r2`
- 학습/유틸 함수들

**V5 10-seed는 실행하지 않은 상태**가 안전합니다 (시간 절약 + 메모리 깨끗).

→ **맨 끝의 Exp 87 셀 3개만 실행** (셀 41~43, 약 2시간 30분)

### 시나리오 B — 런타임 끊김
이전 메모리 복원 패턴 사용. CSV에서 V2, V3, V4 결과 복원 + V4 importance 분석 셀(셀 36)부터 진행.

## V4.5 변경 사항

### 제거 (39개 + 30개 = ~69개)
- N 그룹 (roll30_pct) 12개 — 평균 rank 193.9
- O 그룹 (roll10_pct) 8개 — 평균 rank 206.6
- P 그룹 (since_max/min) 10개 — 평균 rank 192.4
- M 그룹 하위 9개 (rank > 100)
- avg_rank_v4 추가 하위 30개

### 추가 (~25개) — M 패턴 확장
- V (raw → scen_pct): 10개
- W (rolling mean → scen_pct, 5/10/20): 12~15개
- X (vs_cummax → scen_pct): 6개

## PL 전략

**Exp 79 R2 시점 PL 그대로 유지** (검증됨, 누수 없음).


In [1]:
# Cell 1: 패키지 설치 및 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

!pip install -q lightgbm xgboost catboost

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print("설치 완료")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 25.1 MB/s eta 0:00:00
PyTorch: 2.10.0+cu128
CUDA: True
설치 완료


In [2]:
# Cell 2: Import 및 경로 설정
import pandas as pd
import numpy as np
import lightgbm as lgb
from lightgbm import LGBMRegressor
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
from scipy.stats import pearsonr
import os, sys, warnings, gc

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
warnings.filterwarnings('ignore')

DATA_DIR    = '/content/drive/MyDrive/MyDrive/LightGBM'
TRAIN_PATH  = os.path.join(DATA_DIR, 'train.csv')
TEST_PATH   = os.path.join(DATA_DIR, 'test.csv')
LAYOUT_PATH = os.path.join(DATA_DIR, 'layout_info.csv')

# ── 기존 제출 파일들 (PL 합의에 사용) ──
# 사용 가능한 모든 제출 파일을 여기에 나열
SUBMISSION_FILES = {
    '70b':      os.path.join(DATA_DIR, 'submission_70b.csv'),         # LB 10.129
    '72_top2':  os.path.join(DATA_DIR, 'submission_72_top2.csv'),     # LB 10.1287
    '75_g5':    os.path.join(DATA_DIR, 'submission_75_g5.csv'),       # LB 10.1284
    '77_full':  os.path.join(DATA_DIR, 'submission_77_full_only.csv'),# LB 10.1299
}

TARGET = 'avg_delay_minutes_next_30m'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

Device: cuda


In [3]:
# Cell 3: 유틸 함수
def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if pd.api.types.is_numeric_dtype(col_type):
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if   c_min > np.iinfo(np.int8).min  and c_max < np.iinfo(np.int8).max:  df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max: df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max: df[col] = df[col].astype(np.int32)
                else: df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max: df[col] = df[col].astype(np.float32)
                else: df[col] = df[col].astype(np.float64)
    return df

In [4]:
# Cell 4: preprocess_data (Exp 67 원본 — 수정 금지)
def preprocess_data(train, test, layout):
    print("Preprocessing...")
    train = train.merge(layout, on='layout_id', how='left')
    test  = test.merge(layout,  on='layout_id', how='left')

    le = LabelEncoder()
    le.fit(pd.concat([train['layout_type'], test['layout_type']]).astype(str))
    train['layout_type'] = le.transform(train['layout_type'].astype(str))
    test['layout_type']  = le.transform(test['layout_type'].astype(str))

    for df in [train, test]:
        df['robot_utilization']    = df['robot_active']       / (df['robot_total']          + 1e-6)
        df['charger_utilization']  = df['robot_charging']     / (df['charger_count']         + 1e-6)
        df['aisle_pressure']       = df['congestion_score']   / (df['aisle_width_avg']        + 1e-6)
        df['intersection_density'] = df['intersection_count'] / (df['floor_area_sqm']         + 1e-6)
        df['pack_station_pressure']= df['order_inflow_15m']   / (df['pack_station_count']     + 1e-6)
        df['bottleneck_risk']      = df['congestion_score'] * df['intersection_density'] / (df['aisle_width_avg'] + 1e-6)
        df['task_intensity']       = df['order_inflow_15m']   / (df['robot_active']           + 1e-6)

    layout_num_cols = ['aisle_width_avg', 'intersection_count', 'robot_total']
    key_ops         = ['congestion_score', 'robot_active', 'order_inflow_15m']
    for l_col in layout_num_cols:
        for o_col in key_ops:
            train[f'{l_col}_x_{o_col}'] = train[l_col] * train[o_col]
            test[f'{l_col}_x_{o_col}']  = test[l_col]  * test[o_col]

    for col in ['congestion_score', 'low_battery_ratio', 'robot_active']:
        train[f'{col}_vel'] = train.groupby('scenario_id')[col].diff(1)
        test[f'{col}_vel']  = test.groupby('scenario_id')[col].diff(1)

    train['time_idx'] = train.groupby('scenario_id').cumcount()
    test['time_idx']  = test.groupby('scenario_id').cumcount()
    train = train.sort_values(['scenario_id', 'time_idx'])
    test  = test.sort_values(['scenario_id', 'time_idx'])

    SEQ_COLS = [
        'order_inflow_15m','unique_sku_15m','robot_active','robot_idle',
        'robot_charging','battery_mean','battery_std','low_battery_ratio',
        'charge_queue_length','avg_charge_wait','congestion_score',
        'max_zone_density','blocked_path_15m','near_collision_15m',
        'fault_count_15m','avg_recovery_time','task_reassign_15m',
        'replenishment_overlap','pack_utilization','loading_dock_util',
        'staging_area_util','label_print_queue'
    ]

    for col in SEQ_COLS:
        first_tr = train.groupby('scenario_id')[col].transform('first')
        train[f'{col}_vs_start']    = train[col] / (first_tr + 1e-6)
        train[f'{col}_delta_start'] = train[col] - first_tr
        first_ts = test.groupby('scenario_id')[col].transform('first')
        test[f'{col}_vs_start']    = test[col] / (first_ts + 1e-6)
        test[f'{col}_delta_start'] = test[col] - first_ts

    for col in SEQ_COLS:
        if col not in train.columns: continue
        for df in [train, test]:
            prev    = df.groupby('scenario_id')[col].shift(1)
            cum_max = prev.groupby(df['scenario_id']).cummax()
            cum_min = prev.groupby(df['scenario_id']).cummin()
            df[f'{col}_vs_cummax'] = df[col] / (cum_max + 1e-6)
            df[f'{col}_vs_cummin'] = df[col] / (cum_min.abs() + 1e-6)
            cum_range = cum_max - cum_min
            df[f'{col}_position_in_range'] = ((df[col] - cum_min) / (cum_range + 1e-3)).clip(0, 2)

    lag_cols = [
        'congestion_score','fault_count_15m','charge_queue_length',
        'low_battery_ratio','blocked_path_15m','avg_recovery_time',
        'near_collision_15m','pack_utilization'
    ]
    for col in lag_cols:
        if col not in train.columns: continue
        for df in [train, test]:
            for lag in [4,5,6,7]: df[f'{col}_lag{lag}'] = df.groupby('scenario_id')[col].shift(lag)
            prev = df.groupby('scenario_id')[col].shift(1)
            grp  = prev.groupby(df['scenario_id'])
            for w in [7,10,14,20]:
                df[f'{col}_roll{w}_mean'] = grp.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
                df[f'{col}_roll{w}_std']  = grp.rolling(w, min_periods=1).std().reset_index(level=0, drop=True)

    SEQ_COLS_BASE = ['order_inflow_15m','unique_sku_15m','robot_active',
                     'low_battery_ratio','charge_queue_length','congestion_score','fault_count_15m']
    remaining = [c for c in SEQ_COLS_BASE if c not in lag_cols]
    for col in remaining:
        for df in [train, test]:
            for lag in [4,5]: df[f'{col}_lag{lag}'] = df.groupby('scenario_id')[col].shift(lag)
            prev = df.groupby('scenario_id')[col].shift(1)
            grp  = prev.groupby(df['scenario_id'])
            for w in [7,14,20]:
                df[f'{col}_roll{w}_mean'] = grp.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
                df[f'{col}_roll{w}_std']  = grp.rolling(w, min_periods=1).std().reset_index(level=0, drop=True)

    for col in SEQ_COLS:
        for df in [train, test]:
            for lag in [8,10,12,15,20,24]:
                df[f'{col}_lag{lag}'] = df.groupby('scenario_id')[col].shift(lag)

    remaining2 = [c for c in SEQ_COLS if c not in lag_cols]
    for col in remaining2:
        for df in [train, test]:
            prev = df.groupby('scenario_id')[col].shift(1)
            grp  = prev.groupby(df['scenario_id'])
            for w in [14,20]:
                df[f'{col}_roll{w}_mean'] = grp.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
                df[f'{col}_roll{w}_std']  = grp.rolling(w, min_periods=1).std().reset_index(level=0, drop=True)

    tr_new = train['scenario_id'].values != np.roll(train['scenario_id'].values, 1); tr_new[0] = True
    ts_new = test['scenario_id'].values  != np.roll(test['scenario_id'].values,  1); ts_new[0]  = True
    for col in SEQ_COLS_BASE:
        for lag in [1,2]:
            tr_l = train[col].shift(lag).values.copy(); ts_l = test[col].shift(lag).values.copy()
            for l in range(lag):
                tr_l[np.roll(tr_new, l)] = np.nan; ts_l[np.roll(ts_new, l)] = np.nan
            train[f'{col}_lag{lag}'] = tr_l; test[f'{col}_lag{lag}'] = ts_l
        train[f'{col}_exp_mean'] = train.groupby('scenario_id')[col].transform(lambda x: x.shift(1).expanding().mean())
        test[f'{col}_exp_mean']  = test.groupby('scenario_id')[col].transform(lambda x: x.shift(1).expanding().mean())

    train['time_ratio'] = train.groupby('scenario_id')['time_idx'].transform(lambda x: x / (x.max() + 1e-6))
    test['time_ratio']  = test.groupby('scenario_id')['time_idx'].transform(lambda x: x / (x.max() + 1e-6))
    for df in [train, test]:
        df['congestion_ratio'] = df['congestion_score'] / (df['congestion_score_exp_mean'] + 1e-6)
        df['steps_remaining']  = df.groupby('scenario_id')['time_idx'].transform('max') - df['time_idx']

    train.fillna(0, inplace=True); test.fillna(0, inplace=True)
    return reduce_mem_usage(train), reduce_mem_usage(test)

In [5]:
# Cell 5: apply_smoothed_te
def apply_smoothed_te(df_tr, df_val, target_col, k=30):
    global_mean = df_tr[target_col].mean()
    agg = df_tr.groupby('layout_id')[target_col].agg(['mean','std','median','count']).reset_index()
    agg['layout_mean'] = (agg['count'] * agg['mean'] + k * global_mean) / (agg['count'] + k)
    agg.rename(columns={'std':'layout_std','median':'layout_median','count':'layout_count'}, inplace=True)
    df_val = df_val.merge(agg[['layout_id','layout_mean','layout_std','layout_median','layout_count']], on='layout_id', how='left')
    df_tr  = df_tr.merge( agg[['layout_id','layout_mean','layout_std','layout_median','layout_count']], on='layout_id', how='left')
    df_val['layout_mean']   = df_val['layout_mean'].fillna(global_mean)
    df_val['layout_std']    = df_val['layout_std'].fillna(df_tr[target_col].std())
    df_val['layout_median'] = df_val['layout_median'].fillna(df_tr[target_col].median())
    df_val['layout_count']  = df_val['layout_count'].fillna(0)
    return df_tr, df_val, ['layout_mean','layout_std','layout_median','layout_count']

In [6]:
# Cell 6: 공통 설정
zero_imp_features = [
    'charge_queue_length','avg_charge_wait','charge_queue_length_lag2','fault_count_15m_lag2',
    'time_ratio','task_reassign_15m_vs_cummin','low_battery_ratio_vel','low_battery_ratio_lag2',
    'task_reassign_15m','blocked_path_15m_lag8','blocked_path_15m_lag10','near_collision_15m_lag8',
    'near_collision_15m_lag10','fault_count_15m_lag8','fault_count_15m_lag10','avg_recovery_time_lag8',
    'avg_recovery_time_lag10','task_reassign_15m_lag8','task_reassign_15m_lag10','replenishment_overlap_lag8',
    'replenishment_overlap_lag10','robot_charging_lag10','low_battery_ratio_lag8','low_battery_ratio_lag10',
    'charge_queue_length_lag8','charge_queue_length_lag10','avg_charge_wait_lag8','avg_charge_wait_lag10',
    'fault_count_15m_vs_cummax','fault_count_15m_vs_cummin','avg_recovery_time_vs_cummax','task_reassign_15m_vs_cummax',
    'low_battery_ratio_vs_cummax','low_battery_ratio_vs_cummin','charge_queue_length_vs_cummin','avg_charge_wait_vs_cummax',
    'blocked_path_15m_vs_cummax','near_collision_15m_vs_cummin','charge_queue_length_roll14_mean','task_reassign_15m_vs_start',
    'avg_recovery_time_lag5','avg_recovery_time_lag6','avg_recovery_time_lag7','near_collision_15m_lag4',
    'near_collision_15m_lag5','near_collision_15m_lag6','near_collision_15m_lag7','charge_queue_length_roll7_std',
    'charge_queue_length_roll10_mean','low_battery_ratio_lag4','low_battery_ratio_lag5','low_battery_ratio_lag7',
    'blocked_path_15m_lag4','blocked_path_15m_lag5','blocked_path_15m_lag6','blocked_path_15m_lag7',
    'label_print_queue_delta_start','robot_charging_lag15','low_battery_ratio_lag12','low_battery_ratio_lag15',
    'charge_queue_length_lag12','charge_queue_length_lag15','avg_charge_wait_lag12','avg_charge_wait_lag15',
    'congestion_score_lag12','max_zone_density_lag15','blocked_path_15m_lag12','fault_count_15m_lag4',
    'fault_count_15m_lag5','fault_count_15m_lag6','fault_count_15m_lag7','charge_queue_length_lag4',
    'charge_queue_length_lag5','charge_queue_length_lag6','charge_queue_length_lag7','charge_queue_length_roll7_mean',
    'blocked_path_15m_lag15','near_collision_15m_lag12','near_collision_15m_lag15','fault_count_15m_lag12',
    'fault_count_15m_lag15','avg_recovery_time_lag12','avg_recovery_time_lag15','task_reassign_15m_lag12',
    'task_reassign_15m_lag15','replenishment_overlap_lag12','replenishment_overlap_lag15','charge_queue_length_position_in_range',
    'avg_charge_wait_position_in_range','congestion_score_position_in_range','blocked_path_15m_position_in_range',
    'near_collision_15m_position_in_range','fault_count_15m_position_in_range','avg_recovery_time_position_in_range',
    'task_reassign_15m_position_in_range','replenishment_overlap_position_in_range','label_print_queue_position_in_range',
    'replenishment_overlap_vs_cummin','staging_area_util_vs_cummax','battery_mean_position_in_range',
    'low_battery_ratio_position_in_range','label_print_queue_lag15','charge_queue_length_roll20_std',
    'charge_queue_length_vs_start','charge_queue_length_delta_start','avg_charge_wait_vs_start','avg_charge_wait_delta_start'
]

gkf   = GroupKFold(n_splits=5)
seeds = [42, 123, 2026, 777, 1004, 314, 555, 888, 999, 1337]

def target_forward(y): return np.log1p(y)
def target_inverse(p): return np.expm1(p)

# GBDT 파라미터 (Exp 67 확정값)
LGB_PARAMS = {
    'objective': 'huber', 'metric': 'mae', 'boosting_type': 'gbdt',
    'learning_rate': 0.03, 'num_leaves': 127, 'max_depth': -1,
    'min_child_samples': 20, 'subsample': 0.75, 'subsample_freq': 1,
    'colsample_bytree': 0.65, 'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'n_estimators': 3000, 'verbose': -1,
}
XGB_PARAMS = {
    'objective': 'reg:absoluteerror', 'eval_metric': 'mae',
    'learning_rate': 0.03, 'max_depth': 7, 'subsample': 0.75,
    'colsample_bytree': 0.65, 'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'n_estimators': 3000, 'device': 'cuda', 'tree_method': 'hist', 'verbosity': 0,
}
CAT_PARAMS = {
    'loss_function': 'MAE', 'eval_metric': 'MAE',
    'learning_rate': 0.03, 'depth': 7, 'iterations': 3000,
    'l2_leaf_reg': 3.0, 'random_strength': 1.0, 'bagging_temperature': 0.8,
    'task_type': 'GPU', 'devices': '0', 'verbose': 0,
}

print("공통 설정 완료")

공통 설정 완료


In [7]:
# Cell 7: 데이터 로드 + 전처리
print("--- Experiment 79: Pseudo Label 품질 고도화 ---")
train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)
layout    = pd.read_csv(LAYOUT_PATH)

train_layouts  = set(train_raw['layout_id'].unique())
test_layouts   = set(test_raw['layout_id'].unique())
seen_layouts   = train_layouts & test_layouts
unseen_layouts = test_layouts - train_layouts

train, test = preprocess_data(train_raw, test_raw, layout)

EXCLUDE_COLS    = ['ID','layout_id','scenario_id',TARGET,'is_seen','is_pseudo',
                   'pseudo_label','pseudo_std','pseudo_weight']
features_base   = [c for c in train.columns if c not in EXCLUDE_COLS]
features_pruned = [f for f in features_base if f not in zero_imp_features]

train['is_seen'] = train['layout_id'].isin(seen_layouts)
test['is_seen']  = test['layout_id'].isin(seen_layouts)

print(f"피처: {len(features_pruned)}개")
print(f"Train: {len(train)}행")
print(f"Test: {len(test)}행 (Seen {test['is_seen'].sum()}, Unseen {(~test['is_seen']).sum()})")

--- Experiment 79: Pseudo Label 품질 고도화 ---
Preprocessing...
피처: 446개
Train: 250000행
Test: 50000행 (Seen 30000, Unseen 20000)


## [메모리 복원] CSV에서 학습 결과 + PL 변수 복원

런타임이 끊긴 후 V4.5(Exp 87)를 실행하기 위한 셀입니다.

### 복원 흐름

```
[셀 1~6] setup, import, preprocess, TE, 공통 설정
   ↓
[셀 7] 데이터 로드 (train, test, features_pruned)
   ↓
[셀 8 — markdown] 복원 안내
[셀 9 — code] CSV에서 result_v2/v3/v4/r2 + preds + consensus_pl 복원 ⭐
   ↓
[셀 10 — 기존 셀 8] 합의 PL 구성 → 스킵 가능 (이미 위에서 만들었음)
[셀 11~12] build_pseudo_dataset 함수 정의 (필수 실행)
[셀 13] train_combined_r2 alias (필수 실행)
[셀 14] train_gbdt_round 함수 정의 (필수 실행)
   ↓
[셀 15~19] R1, R2 학습 → 스킵 (result_r2 복원됨)
[셀 20] V2 importance 분석
[셀 21] (마크다운)
[셀 22] V2 피처 생성 → features_v2 (339개)
[셀 23, 24] V2 학습 → 스킵 (result_v2 복원됨)
   ↓
[셀 26] V3 importance 분석
[셀 27] V3 피처 생성 → features_v3 (277개)
[셀 28, 29] V3 학습 → 스킵 (result_v3 복원됨)
   ↓
[셀 31] V4 importance 분석
[셀 32] V4 피처 생성 → features_v4 (219개)
[셀 33, 34] V4 학습 → 스킵 (result_v4 복원됨)
   ↓
[셀 38] V4 importance 재분석 → avg_rank_v4 ⭐ V4.5에 필수
[셀 40, 41] V5 학습 → 스킵 (V4 못 이김)
   ↓
[셀 43~45] V4.5 본 실행 ⭐
```

### 사전 체크리스트

Drive `/content/drive/MyDrive/MyDrive/LightGBM/`에 다음 파일 필수:

```
필수 (PL 합의용):
  submission_70b.csv
  submission_72_top2.csv
  submission_75_g5.csv
  submission_77_full_only.csv

필수 (학습 결과 복원):
  submission_79_r2.csv  ⭐
  submission_82_v2.csv  ⭐
  submission_84_v3.csv  ⭐
  submission_85_v4.csv  ⭐ (V4.5에 가장 중요)
```

### 시간 절약

| 단계 | 정상 학습 시 | 복원 시 |
|------|-------------|---------|
| Setup | 5분 | 5분 |
| 데이터 로드 | 3분 | 3분 |
| **메모리 복원** | — | **3분** |
| R1, R2 학습 | 4시간 | 0분 (스킵) |
| V2 importance + 피처 | 12분 | 12분 |
| V2 학습 | 2시간 | 0분 (스킵) |
| V3 importance + 피처 | 12분 | 12분 |
| V3 학습 | 2시간 | 0분 (스킵) |
| V4 importance + 피처 | 15분 | 15분 |
| V4 학습 | 2시간 | 0분 (스킵) |
| V4 importance 재분석 | 10분 | 10분 |
| V4.5 본 실행 | 2시간 30분 | 2시간 30분 |
| **합계** | **~13.5시간** | **~3.5시간** |

**약 10시간 절약**.


In [ ]:
# Cell 7.5: 메모리 복원 — CSV에서 result_v2, v3, v4, r2 + preds + consensus_pl 복원
# 런타임 끊긴 후 Exp 87 V4.5 실행 사전 셀
# 이 셀이 실행되면 다음 셀(Cell 8 — 합의 PL 구성)은 스킵 가능

print("=" * 60)
print("[메모리 복원] CSV에서 학습 결과 + PL 변수 복원")
print("=" * 60)

# ── 사전조건 체크 ──
assert 'train' in dir() and 'test' in dir(), "train, test 없음 — 셀 7을 먼저 실행"
assert 'TARGET' in dir(), "TARGET 없음 — 셀 6을 먼저 실행"
assert 'DATA_DIR' in dir(), "DATA_DIR 없음 — 셀 6을 먼저 실행"

# ── SUBMISSION_FILES 자체 정의 (셀 8 의존성 제거) ──
SUBMISSION_FILES = {
    '70b':     os.path.join(DATA_DIR, 'submission_70b.csv'),
    '72_top2': os.path.join(DATA_DIR, 'submission_72_top2.csv'),
    '75_g5':   os.path.join(DATA_DIR, 'submission_75_g5.csv'),
    '77_full': os.path.join(DATA_DIR, 'submission_77_full_only.csv'),
}
print(f"\n  SUBMISSION_FILES 정의됨: {list(SUBMISSION_FILES.keys())}")

# ── ① 합의 PL 모델 복원 ──
print(f"\n[1/4] 합의 PL 모델 복원")
preds = {}
for name, path in SUBMISSION_FILES.items():
    if os.path.exists(path):
        df = pd.read_csv(path)
        preds[name] = df.set_index('ID').loc[test['ID'].values][TARGET].values
        print(f"  ✅ preds['{name}']")
    else:
        print(f"  ⚠️ preds['{name}'] 파일 없음: {path}")

if not preds:
    raise FileNotFoundError("PL 모델 CSV가 하나도 없습니다 — Drive 확인 필요")

# ── ② 학습 결과 복원 (R2, V2, V3, V4) ──
print(f"\n[2/4] 학습 결과 복원")

results_to_restore = [
    ('submission_79_r2.csv', 'result_r2', '10.1201', 'Exp 79 R2'),
    ('submission_82_v2.csv', 'result_v2', '10.0813', 'Exp 82 V2'),
    ('submission_84_v3.csv', 'result_v3', '10.0737', 'Exp 84 V3'),
    ('submission_85_v4.csv', 'result_v4', '10.0480', 'Exp 85 V4 (PB)'),
]

for fname, var_name, lb, desc in results_to_restore:
    path = os.path.join(DATA_DIR, fname)
    if os.path.exists(path):
        pred = pd.read_csv(path).set_index('ID').loc[test['ID'].values][TARGET].values
        globals()[var_name] = {'test_pred': pred}
        print(f"  ✅ {var_name} 복원 ({desc}, LB {lb})")
    else:
        print(f"  ⚠️ {fname} 없음 — {var_name} 복원 실패")

# V4 필수 체크
assert 'result_v4' in dir(), \
    "❌ result_v4 복원 실패 — submission_85_v4.csv가 Drive에 있어야 합니다"

# ── ③ consensus_pl, pred_std 재구성 (Exp 79 R2 PL — V4.5의 학습 데이터) ──
print(f"\n[3/4] consensus_pl 재구성 (Exp 79 R2 PL)")

LB_SCORES = {'70b': 10.129, '72_top2': 10.1287, '75_g5': 10.1284, '77_full': 10.1299}
available_lbs = {k: v for k, v in LB_SCORES.items() if k in preds}

if not available_lbs:
    raise ValueError("PL 가중치 계산 불가 — preds dict가 비어있음")

inv_lb = {k: 1.0 / v for k, v in available_lbs.items()}
total_inv = sum(inv_lb.values())
weights = {k: v / total_inv for k, v in inv_lb.items()}

print(f"  PL 가중치 (LB inverse):")
for name, w in weights.items():
    print(f"    {name:12s}: {w:.4f}")

consensus_pl = np.zeros(len(test))
for name, w in weights.items():
    consensus_pl += w * preds[name]

pred_matrix = np.column_stack([preds[name] for name in preds])
pred_std = pred_matrix.std(axis=1)

# ── ④ test에 pseudo_label, is_seen 주입 ──
print(f"\n[4/4] test에 pseudo 변수 주입")

test['pseudo_label'] = consensus_pl
test['pseudo_std']   = pred_std

# is_seen
train_layouts = set(train['layout_id'].unique())
test_layouts  = set(test['layout_id'].unique())
seen_layouts  = train_layouts & test_layouts

test['is_seen'] = test['layout_id'].isin(seen_layouts)
print(f"  test seen 비율: {test['is_seen'].mean():.2%}")
print(f"  consensus_pl mean: {consensus_pl.mean():.4f}")
print(f"  consensus_pl std:  {consensus_pl.std():.4f}")
print(f"  pred_std mean:     {pred_std.mean():.4f}")

# ── 완료 ──
print(f"\n{'='*60}")
print(f"[메모리 복원 완료]")
print(f"{'='*60}")
print(f"\n복원된 변수:")
print(f"  - preds: {list(preds.keys())}")
for v in ['result_r2', 'result_v2', 'result_v3', 'result_v4']:
    mark = '✅' if v in dir() else '❌'
    print(f"  - {v}: {mark}")
print(f"  - consensus_pl, pred_std")
print(f"  - test['pseudo_label'], test['pseudo_std'], test['is_seen']")
print(f"\n다음 실행 순서:")
print(f"  → 셀 8 (합의 PL 구성)도 동일한 작업 수행하므로 스킵 가능")
print(f"  → 셀 9 (build_pseudo_dataset 함수 정의) 실행")
print(f"  → 셀 10 (build_pseudo_dataset 함수 정의 본체) 실행")
print(f"  → 셀 11 (train_combined_r2 alias 생성) 실행")
print(f"  → 셀 12 (train_gbdt_round 함수 정의) 실행")
print(f"  → 셀 13~17 스킵 (R1, R2 학습 — result_r2 복원됨)")
print(f"  → 셀 18 → 셀 19 → 셀 20 (V2 importance + features_v2)")
print(f"  → 셀 21, 22 스킵 (V2 학습 — result_v2 복원됨)")
print(f"  → V3, V4 동일 패턴")
print(f"  → 셀 36 (V4 importance 재분석 — V4.5에 필수)")
print(f"  → 셀 41~43 (V4.5 본 실행)")


In [ ]:
# Cell 8: [Stage 0] 다중 모델 합의 PL 구성
#
# 핵심 아이디어:
#   1) 여러 제출 파일의 가중 평균 → 개별 noise 상쇄
#   2) 모델 간 예측 분산(std) → 신뢰도 가중치
#      - std 작은 행: 모델들이 동의 → 높은 weight
#      - std 큰 행: 모델들이 불일치 → 낮은 weight
#   3) 기존 전략C의 이진 필터도 유지 (극단값/저분산 layout)

print("[Stage 0] 다중 모델 합의 Pseudo Label 구성")
print("="*60)

# ── 제출 파일 로드 ──
preds = {}
for name, path in SUBMISSION_FILES.items():
    if os.path.exists(path):
        df = pd.read_csv(path)
        preds[name] = df.set_index('ID').loc[test['ID'].values][TARGET].values
        print(f"  ✅ {name}: 로드 완료 (mean={preds[name].mean():.4f})")
    else:
        print(f"  ❌ {name}: 파일 없음 — 스킵")

n_models = len(preds)
print(f"\n  사용 가능한 모델: {n_models}개")

if n_models < 2:
    print("  ⚠️ 모델이 2개 미만. 합의 PL 효과 제한적.")
    print("  → 가장 좋은 단일 모델을 PL로 사용합니다.")

# ── 모델 간 상관관계 ──
pred_names = list(preds.keys())
print(f"\n  [모델 간 상관관계]")
for i in range(len(pred_names)):
    for j in range(i+1, len(pred_names)):
        r, _ = pearsonr(preds[pred_names[i]], preds[pred_names[j]])
        print(f"    {pred_names[i]} ↔ {pred_names[j]}: r={r:.4f}")

# ── LB 기반 가중 평균 (Inverse-LB weighting) ──
# LB가 낮을수록 좋은 모델 → 높은 가중치
LB_SCORES = {
    '70b':     10.129,
    '72_top2': 10.1287,
    '75_g5':   10.1284,
    '77_full': 10.1299,
}

# 사용 가능한 모델만 필터
available_lbs = {k: v for k, v in LB_SCORES.items() if k in preds}
inv_lb = {k: 1.0 / v for k, v in available_lbs.items()}
total_inv = sum(inv_lb.values())
weights = {k: v / total_inv for k, v in inv_lb.items()}

print(f"\n  [LB 기반 가중치]")
for k, w in weights.items():
    print(f"    {k}: LB={available_lbs[k]:.4f} → weight={w:.4f}")

# ── 합의 PL 생성 ──
consensus_pl = np.zeros(len(test))
for name, w in weights.items():
    consensus_pl += w * preds[name]

# ── 모델 간 분산 (신뢰도 지표) ──
pred_matrix = np.column_stack([preds[name] for name in preds])
pred_std = pred_matrix.std(axis=1)    # 행별 표준편차
pred_mean_abs_dev = np.abs(pred_matrix - pred_matrix.mean(axis=1, keepdims=True)).mean(axis=1)

test['pseudo_label'] = consensus_pl
test['pseudo_std']   = pred_std

print(f"\n  [합의 PL 통계]")
print(f"    mean: {consensus_pl.mean():.4f}")
print(f"    std:  {consensus_pl.std():.4f}")
print(f"  [모델 간 분산 (pred_std)]")
print(f"    mean: {pred_std.mean():.4f}")
print(f"    median: {np.median(pred_std):.4f}")
print(f"    p95: {np.percentile(pred_std, 95):.4f}")
print(f"    max: {pred_std.max():.4f}")

# Seen vs Unseen 분산 비교
seen_mask = test['is_seen'].values
print(f"\n  [Seen vs Unseen 분산]")
print(f"    Seen   mean_std: {pred_std[seen_mask].mean():.4f}")
print(f"    Unseen mean_std: {pred_std[~seen_mask].mean():.4f}")

print("\n[Stage 0 완료]")

In [9]:
# Cell 9: Confidence-weighted Pseudo 데이터 구성 함수
#
# [기존 전략C vs 신규 전략D 비교]
#
# 전략C (기존):
#   - 이진 필터: 극단값 제외, 저분산 Unseen layout 제외
#   - 포함된 행은 모두 동일 weight (Seen=1.0, Unseen=0.2)
#
# 전략D (신규):
#   - 전략C의 이진 필터 유지 (안전장치)
#   - 추가로 모델 간 분산 기반 연속 가중치
#   - 분산 작은 행(모델 동의) → weight 1.0
#   - 분산 큰 행(모델 불일치) → weight 0.2~0.5
#   - → 기존보다 세밀한 가중치 부여

def build_pseudo_dataset(train, test, consensus_col='pseudo_label',
                         std_col='pseudo_std', strategy='D'):
    """
    Pseudo Label 데이터셋 구성

    strategy:
      'C' — 기존 이진 필터 (Exp 70b와 동일)
      'D' — 이진 필터 + confidence weighting (신규)

    Returns:
      train_combined: 원본 train + pseudo test
      pseudo_stats: 통계 정보
    """
    train_std = train[TARGET].std()
    train_q01 = train[TARGET].quantile(0.01)
    train_q99 = train[TARGET].quantile(0.99)

    # ── 이진 필터 (전략C, D 공통) ──
    unseen_test    = test[~test['is_seen']]
    layout_std_map = unseen_test.groupby('layout_id')[consensus_col].std()
    low_var_lay    = layout_std_map[layout_std_map < train_std * 0.3].index.tolist()
    mask_low_var   = test['layout_id'].isin(low_var_lay) & ~test['is_seen']
    mask_extreme   = (test[consensus_col] < train_q01) | (test[consensus_col] > train_q99)

    pseudo_mask = (
        (test['is_seen'] & ~mask_extreme) |
        (~test['is_seen'] & ~mask_low_var & ~mask_extreme)
    )

    pseudo_set = test[pseudo_mask].copy()
    pseudo_set[TARGET]      = pseudo_set[consensus_col]
    pseudo_set['is_pseudo'] = True

    # ── Confidence weight 계산 (전략D만) ──
    if strategy == 'D' and std_col in test.columns:
        # 분산 기반 신뢰도: std가 작을수록 높은 weight
        pseudo_std_vals = pseudo_set[std_col].values

        # 정규화: [0, 1] → [w_min, 1.0]
        # p90 기준으로 정규화 (극단적 outlier에 강건)
        std_p90 = np.percentile(pseudo_std_vals, 90)
        std_normalized = np.clip(pseudo_std_vals / (std_p90 + 1e-6), 0, 2)

        # confidence: 1 - normalized_std → [0, 1]
        # w_min=0.2로 바운드 (너무 낮으면 pseudo 효과 소멸)
        w_min = 0.2
        confidence = np.clip(1.0 - std_normalized, 0, 1)
        pseudo_weight = w_min + (1.0 - w_min) * confidence

        pseudo_set['pseudo_weight'] = pseudo_weight
    else:
        pseudo_set['pseudo_weight'] = 1.0  # 전략C: 동일 weight

    # ── 결합 ──
    shared_cols = [c for c in train.columns if c in pseudo_set.columns]
    train_c = train.copy()
    train_c['is_pseudo'] = False
    train_c['pseudo_weight'] = 1.0  # 원본은 항상 weight=1.0

    if 'pseudo_weight' not in shared_cols:
        shared_cols.append('pseudo_weight')

    train_combined = pd.concat(
        [train_c[shared_cols], pseudo_set[shared_cols]], ignore_index=True)
    train_combined['is_pseudo'] = train_combined['is_pseudo'].fillna(False)

    # ── 통계 ──
    n_pseudo = pseudo_mask.sum()
    n_pseudo_seen   = (pseudo_mask & test['is_seen']).sum()
    n_pseudo_unseen = (pseudo_mask & ~test['is_seen']).sum()
    n_excluded = len(test) - n_pseudo

    stats = {
        'n_pseudo': n_pseudo,
        'n_pseudo_seen': n_pseudo_seen,
        'n_pseudo_unseen': n_pseudo_unseen,
        'n_excluded': n_excluded,
        'n_low_var': mask_low_var.sum(),
        'n_extreme': mask_extreme.sum(),
        'low_var_layouts': low_var_lay,
    }

    if strategy == 'D':
        stats['weight_mean'] = pseudo_set['pseudo_weight'].mean()
        stats['weight_median'] = pseudo_set['pseudo_weight'].median()
        stats['weight_min'] = pseudo_set['pseudo_weight'].min()

    return train_combined, stats


def make_sample_weight_v2(df, w_orig=1.0, w_seen_base=1.0, w_unseen_base=0.2):
    """
    전략D용 sample weight:
      원본 train: w_orig
      Pseudo Seen:   w_seen_base × pseudo_weight (confidence)
      Pseudo Unseen: w_unseen_base × pseudo_weight (confidence)
    """
    is_pseudo = df['is_pseudo'].fillna(False).values
    is_seen   = df['is_seen'].values
    pw        = df['pseudo_weight'].fillna(1.0).values

    return np.where(
        ~is_pseudo, w_orig,
        np.where(is_seen, w_seen_base * pw, w_unseen_base * pw)
    )


print("Pseudo 구성 함수 정의 완료")

Pseudo 구성 함수 정의 완료


In [11]:
def build_pseudo_dataset(train, test, consensus_col='pseudo_label',
                         std_col='pseudo_std', strategy='D'):
    """
    Pseudo Label 데이터셋 구성

    strategy:
      'C' — 기존 이진 필터 (Exp 70b와 동일)
      'D' — 이진 필터 + confidence weighting (신규)

    Returns:
      train_combined: 원본 train + pseudo test
      pseudo_stats: 통계 정보
    """
    train_std = train[TARGET].std()
    train_q01 = train[TARGET].quantile(0.01)
    train_q99 = train[TARGET].quantile(0.99)

    # ── 이진 필터 (전략C, D 공통) ──
    unseen_test    = test[~test['is_seen']]
    layout_std_map = unseen_test.groupby('layout_id')[consensus_col].std()
    low_var_lay    = layout_std_map[layout_std_map < train_std * 0.3].index.tolist()
    mask_low_var   = test['layout_id'].isin(low_var_lay) & ~test['is_seen']
    mask_extreme   = (test[consensus_col] < train_q01) | (test[consensus_col] > train_q99)

    pseudo_mask = (
        (test['is_seen'] & ~mask_extreme) |
        (~test['is_seen'] & ~mask_low_var & ~mask_extreme)
    )

    pseudo_set = test[pseudo_mask].copy()
    pseudo_set[TARGET]      = pseudo_set[consensus_col]
    pseudo_set['is_pseudo'] = True

    # ── Confidence weight 계산 (전략D만) ──
    if strategy == 'D' and std_col in test.columns:
        # 분산 기반 신뢰도: std가 작을수록 높은 weight
        pseudo_std_vals = pseudo_set[std_col].values

        # 정규화: [0, 1] → [w_min, 1.0]
        # p90 기준으로 정규화 (극단적 outlier에 강건)
        std_p90 = np.percentile(pseudo_std_vals, 90)
        std_normalized = np.clip(pseudo_std_vals / (std_p90 + 1e-6), 0, 2)

        # confidence: 1 - normalized_std → [0, 1]
        # w_min=0.2로 바운드 (너무 낮으면 pseudo 효과 소멸)
        w_min = 0.2
        confidence = np.clip(1.0 - std_normalized, 0, 1)
        pseudo_weight = w_min + (1.0 - w_min) * confidence

        pseudo_set['pseudo_weight'] = pseudo_weight
    else:
        pseudo_set['pseudo_weight'] = 1.0  # 전략C: 동일 weight

    # ── 결합 ──
    # 원본 train 데이터프레임의 복사본에 'is_pseudo'와 'pseudo_weight' 컬럼을 추가
    train_c = train.copy()
    train_c['is_pseudo'] = False
    train_c['pseudo_weight'] = 1.0  # 원본 train 데이터의 가중치는 1.0

    # 두 데이터프레임 (train_c와 pseudo_set)이 공통으로 가지는 컬럼들을 찾아 concat에 사용
    # 이렇게 하면 'is_pseudo'와 'pseudo_weight'가 자동으로 포함됨
    shared_cols = list(set(train_c.columns) & set(pseudo_set.columns))

    train_combined = pd.concat(
        [train_c[shared_cols], pseudo_set[shared_cols]], ignore_index=True)

    # 'is_pseudo' 컬럼은 이미 위에서 train_c와 pseudo_set에 명시적으로 할당되었고,
    # shared_cols에 포함되어 concat되었으므로, 이 시점에는 NaN이 없어야 함.
    # 따라서 `train_combined['is_pseudo'] = train_combined['is_pseudo'].fillna(False)`는 필요 없음.
    # train_combined['is_pseudo'] = train_combined['is_pseudo'].fillna(False) # <--- 이 줄은 삭제

    # ── 통계 ──
    n_pseudo = pseudo_mask.sum()
    n_pseudo_seen   = (pseudo_mask & test['is_seen']).sum()
    n_pseudo_unseen = (pseudo_mask & ~test['is_seen']).sum()
    n_excluded = len(test) - n_pseudo

    stats = {
        'n_pseudo': n_pseudo,
        'n_pseudo_seen': n_pseudo_seen,
        'n_pseudo_unseen': n_pseudo_unseen,
        'n_excluded': n_excluded,
        'n_low_var': mask_low_var.sum(),
        'n_extreme': mask_extreme.sum(),
        'low_var_layouts': low_var_lay,
    }

    if strategy == 'D':
        stats['weight_mean'] = pseudo_set['pseudo_weight'].mean()
        stats['weight_median'] = pseudo_set['pseudo_weight'].median()
        stats['weight_min'] = pseudo_set['pseudo_weight'].min()

    return train_combined, stats


print("[전략 C vs D 비교]")
print("="*60)

# 전략 C (기존 — 비교 기준)
tc_c, stats_c = build_pseudo_dataset(train, test, strategy='C')
print(f"\n  전략 C (기존 이진 필터):")
print(f"    Pseudo 포함: {stats_c['n_pseudo']}행 (Seen {stats_c['n_pseudo_seen']}, Unseen {stats_c['n_pseudo_unseen']})")
print(f"    제외: {stats_c['n_excluded']}행 (저분산 {stats_c['n_low_var']}, 극단값 {stats_c['n_extreme']})")
print(f"    저분산 layout: {stats_c['low_var_layouts']}")

# 전략 D (신규 — confidence weighting)
tc_d, stats_d = build_pseudo_dataset(train, test, strategy='D')
print(f"\n  전략 D (이진 필터 + confidence weight):")
print(f"    Pseudo 포함: {stats_d['n_pseudo']}행 (동일 필터)")
print(f"    Weight 분포: mean={stats_d['weight_mean']:.4f}, median={stats_d['weight_median']:.4f}, min={stats_d['weight_min']:.4f}")

# 전략 D 사용 (합의 PL + confidence weight)
train_combined = tc_d
del tc_c  # 메모리 절약
gc.collect()

print(f"\n  ✅ 전략 D 채택: 합의 PL + confidence weighting")
print(f"  통합 train: {len(train_combined)}행 (원본 {len(train)} + pseudo {stats_d['n_pseudo']})")

[전략 C vs D 비교]

  전략 C (기존 이진 필터):
    Pseudo 포함: 47600행 (Seen 30000, Unseen 17600)
    제외: 2400행 (저분산 2400, 극단값 0)
    저분산 layout: ['WH_019', 'WH_143', 'WH_234', 'WH_241', 'WH_242', 'WH_281']

  전략 D (이진 필터 + confidence weight):
    Pseudo 포함: 47600행 (동일 필터)
    Weight 분포: mean=0.7021, median=0.7927, min=0.2000

  ✅ 전략 D 채택: 합의 PL + confidence weighting
  통합 train: 297600행 (원본 250000 + pseudo 47600)


In [12]:
# train_combined_r2 alias 생성 (Exp 84/85가 이 변수명을 사용)
train_combined_r2 = train_combined.copy()
print(f"✅ train_combined_r2 생성: {len(train_combined_r2)}행")

✅ train_combined_r2 생성: 297600행


In [13]:
# Cell 11: GBDT 학습 함수 (전략D weight 적용)

def train_gbdt_round(train_combined, test, features, seeds, round_name="R1"):
    """
    GBDT 3종 × N-seed 앙상블 (전략D weight 적용)
    """
    orig_mask = ~train_combined['is_pseudo'].fillna(False).values
    y_orig    = train_combined[TARGET].values[orig_mask]
    n_orig    = orig_mask.sum()
    n_test    = len(test)

    oof  = {k: np.zeros(n_orig) for k in ['lgb','xgb','cat']}
    tprd = {k: np.zeros(n_test) for k in ['lgb','xgb','cat']}

    n_seeds = len(seeds)
    total_models = n_seeds * 5 * 3
    model_count = 0

    for seed_idx, seed in enumerate(seeds):
        print(f"\n{'='*25} {round_name} Seed {seed} ({seed_idx+1}/{n_seeds}) {'='*25}")

        oof_seed  = {k: np.zeros(len(train_combined)) for k in ['lgb','xgb','cat']}
        test_seed = {k: np.zeros(n_test) for k in ['lgb','xgb','cat']}

        for fold, (tr_idx, val_idx) in enumerate(
                gkf.split(train_combined, train_combined[TARGET],
                          groups=train_combined['layout_id'])):

            X_tr_raw  = train_combined.iloc[tr_idx].copy()
            y_tr_raw  = train_combined.iloc[tr_idx][TARGET].values
            sw_tr     = make_sample_weight_v2(train_combined.iloc[tr_idx])
            X_val_raw = train_combined.iloc[val_idx].copy()
            y_val_raw = train_combined.iloc[val_idx][TARGET].values

            # CV-safe TE
            X_tr_raw, X_val_raw, te_cols = apply_smoothed_te(
                X_tr_raw, X_val_raw, TARGET, k=30)
            _, X_te_raw, _ = apply_smoothed_te(
                X_tr_raw, test.copy(), TARGET, k=30)

            X_tr_raw.fillna(0, inplace=True)
            X_val_raw.fillna(0, inplace=True)
            X_te_raw.fillna(0, inplace=True)

            feats = features + te_cols
            y_tr_t  = target_forward(y_tr_raw)
            y_val_t = target_forward(y_val_raw)

            # ── LightGBM ──
            lgb_p = {**LGB_PARAMS, 'random_state': seed}
            m_lgb = LGBMRegressor(**lgb_p)
            m_lgb.fit(
                X_tr_raw[feats], y_tr_t, sample_weight=sw_tr,
                eval_set=[(X_val_raw[feats], y_val_t)],
                callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)]
            )
            p_val_lgb  = np.maximum(target_inverse(m_lgb.predict(X_val_raw[feats])), 0)
            p_test_lgb = np.maximum(target_inverse(m_lgb.predict(X_te_raw[feats])), 0)
            oof_seed['lgb'][val_idx]  = p_val_lgb
            test_seed['lgb']         += p_test_lgb / 5
            model_count += 1

            # ── XGBoost ──
            xgb_p = {**XGB_PARAMS, 'random_state': seed}
            m_xgb = xgb.XGBRegressor(**xgb_p)
            m_xgb.fit(
                X_tr_raw[feats], y_tr_t, sample_weight=sw_tr,
                eval_set=[(X_val_raw[feats], y_val_t)], verbose=False
            )
            p_val_xgb  = np.maximum(target_inverse(m_xgb.predict(X_val_raw[feats])), 0)
            p_test_xgb = np.maximum(target_inverse(m_xgb.predict(X_te_raw[feats])), 0)
            oof_seed['xgb'][val_idx]  = p_val_xgb
            test_seed['xgb']         += p_test_xgb / 5
            model_count += 1

            # ── CatBoost ──
            cat_p = {**CAT_PARAMS, 'random_seed': seed}
            m_cat = CatBoostRegressor(**cat_p)
            m_cat.fit(
                X_tr_raw[feats], y_tr_t, sample_weight=sw_tr,
                eval_set=(X_val_raw[feats], y_val_t),
                early_stopping_rounds=100, verbose=0
            )
            p_val_cat  = np.maximum(target_inverse(m_cat.predict(X_val_raw[feats])), 0)
            p_test_cat = np.maximum(target_inverse(m_cat.predict(X_te_raw[feats])), 0)
            oof_seed['cat'][val_idx]  = p_val_cat
            test_seed['cat']         += p_test_cat / 5
            model_count += 1

            fold_mae_lgb = mean_absolute_error(y_val_raw, p_val_lgb)
            fold_mae_xgb = mean_absolute_error(y_val_raw, p_val_xgb)
            fold_mae_cat = mean_absolute_error(y_val_raw, p_val_cat)
            print(f"  F{fold+1} LGB={fold_mae_lgb:.4f} XGB={fold_mae_xgb:.4f} CAT={fold_mae_cat:.4f}"
                  f"  [{model_count}/{total_models}]", flush=True)

            del m_lgb, m_xgb, m_cat; gc.collect()

        orig_indices = np.where(orig_mask)[0]
        for k in ['lgb','xgb','cat']:
            oof[k]  += oof_seed[k][orig_indices] / n_seeds
            tprd[k] += test_seed[k] / n_seeds

        seed_ens = (oof_seed['lgb'][orig_mask] + oof_seed['xgb'][orig_mask] + oof_seed['cat'][orig_mask]) / 3
        print(f"  Seed {seed} Ens OOF: {mean_absolute_error(y_orig, seed_ens):.4f}")

    # ── Inverse MAE 가중 앙상블 ──
    mae_lgb = mean_absolute_error(y_orig, oof['lgb'])
    mae_xgb = mean_absolute_error(y_orig, oof['xgb'])
    mae_cat = mean_absolute_error(y_orig, oof['cat'])
    inv_mae = np.array([1/mae_lgb, 1/mae_xgb, 1/mae_cat])
    weights = inv_mae / inv_mae.sum()

    oof_ens  = weights[0]*oof['lgb']  + weights[1]*oof['xgb']  + weights[2]*oof['cat']
    test_ens = weights[0]*tprd['lgb'] + weights[1]*tprd['xgb'] + weights[2]*tprd['cat']
    ens_mae  = mean_absolute_error(y_orig, oof_ens)

    print(f"\n  {round_name} LGB={mae_lgb:.4f} XGB={mae_xgb:.4f} CAT={mae_cat:.4f}")
    print(f"  {round_name} weights: LGB={weights[0]:.4f} XGB={weights[1]:.4f} CAT={weights[2]:.4f}")
    print(f"  {round_name} Ensemble OOF MAE: {ens_mae:.4f}")

    return {
        'test_pred': np.maximum(test_ens, 0),
        'oof_pred': oof_ens,
        'oof_mae': ens_mae,
        'weights': weights,
        'test_preds_individual': {'lgb': tprd['lgb'], 'xgb': tprd['xgb'], 'cat': tprd['cat']},
    }

print("GBDT 학습 함수 정의 완료")

GBDT 학습 함수 정의 완료


In [ ]:
# Cell 12: [Stage 1] Round 1 — 합의 PL + Confidence Weight로 GBDT 학습
#
# 변경점 (기존 70b PL 대비):
#   1) PL 값: 70b 단독 → 다중 모델 가중 평균 (noise 감소)
#   2) Weight: 이진(1.0/0.2) → confidence 기반 연속 (세밀)
# 파이프라인 나머지는 Exp 67/70b와 100% 동일
#
# 소요 시간: 약 2시간

print("\n" + "="*70)
print(f"  [Stage 1] Round 1 — 합의 PL + Confidence Weight")
print(f"  PL: {n_models}개 모델 LB-weighted 평균")
print(f"  Weight: 전략D (confidence)")
print(f"  피처: {len(features_pruned)}개 + TE 4개")
print(f"  Train: {len(train_combined)}행")
print("="*70)

result_r1 = train_gbdt_round(
    train_combined, test, features_pruned, seeds, round_name="R1")

print(f"\n[Stage 1 완료] R1 OOF MAE: {result_r1['oof_mae']:.4f}")
print(f"  기존 Exp 70b OOF: 8.6935")
print(f"  기존 Exp 77 Full OOF: 8.6886")
print(f"  차이 (R1 - 77Full): {result_r1['oof_mae'] - 8.6886:+.4f}")

In [ ]:
# Cell 13: Round 1 제출 파일 저장 + 기존 best와 블렌딩

print("[Round 1 제출 파일 생성]")
print("="*60)

# 기존 best 로드
pred_72 = pd.read_csv(SUBMISSION_FILES['72_top2']).set_index('ID').loc[test['ID'].values][TARGET].values

try:
    pred_75 = pd.read_csv(SUBMISSION_FILES['75_g5']).set_index('ID').loc[test['ID'].values][TARGET].values
    has_75 = True
except:
    has_75 = False

saved_files = []

# ── R1 단독 ──
fname = 'submission_79_r1.csv'
fpath = os.path.join(DATA_DIR, fname)
pd.DataFrame({'ID': test['ID'], TARGET: result_r1['test_pred']}).to_csv(fpath, index=False)
saved_files.append(('R1 단독', fname))
print(f"  ✅ {fname} — Round 1 단독")

# ── R1 + 72_top2 블렌딩 ──
for alpha in [0.3, 0.5, 0.7]:
    blend = alpha * result_r1['test_pred'] + (1 - alpha) * pred_72
    blend = np.maximum(blend, 0)
    fname = f'submission_79_r1_b{int(alpha*10)}.csv'
    fpath = os.path.join(DATA_DIR, fname)
    pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(fpath, index=False)
    saved_files.append((f'R1×{int(alpha*100)}% + 72×{int((1-alpha)*100)}%', fname))
    print(f"  ✅ {fname} — R1 {int(alpha*100)}% + 72_top2 {int((1-alpha)*100)}%")

# ── R1 + 75_g5 블렌딩 ──
if has_75:
    for alpha in [0.3, 0.5, 0.7]:
        blend = alpha * result_r1['test_pred'] + (1 - alpha) * pred_75
        blend = np.maximum(blend, 0)
        fname = f'submission_79_r1_75b{int(alpha*10)}.csv'
        fpath = os.path.join(DATA_DIR, fname)
        pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(fpath, index=False)
        saved_files.append((f'R1×{int(alpha*100)}% + 75×{int((1-alpha)*100)}%', fname))
        print(f"  ✅ {fname} — R1 {int(alpha*100)}% + 75_g5 {int((1-alpha)*100)}%")

# ── 72_top2 대비 상관관계 ──
corr_r1_72, _ = pearsonr(result_r1['test_pred'], pred_72)
print(f"\n  R1 ↔ 72_top2 상관관계: {corr_r1_72:.4f}")
print(f"  R1 ↔ 72_top2 mean diff: {np.abs(result_r1['test_pred'] - pred_72).mean():.4f}")

print(f"\n총 {len(saved_files)}개 파일 저장")
print(f"\n📋 제출 순서:")
print(f"  1순위: submission_79_r1.csv (R1 단독 — 합의PL 효과 순수 측정)")
print(f"  2순위: submission_79_r1_b5.csv (R1 50% + 72_top2 50% — 안전)")
if has_75:
    print(f"  3순위: submission_79_r1_75b5.csv (R1 50% + 75_g5 50%)")

In [ ]:
# Cell 14: [Stage 2] Round 2 — R1 예측으로 PL 갱신 후 재학습
#
# R1의 test 예측을 새 PL로 사용 → 합의 PL을 한단계 더 개선
# R1 예측 + 기존 합의 PL을 다시 평균하여 더 정제된 PL 생성
#
# ⚠️ Stage 1 LB 결과를 확인한 후에 실행하는 것을 권장
# R1 LB가 기존보다 좋으면 → R2 진행
# R1 LB가 기존보다 나쁘면 → R2 스킵, 블렌딩만 탐색
#
# 소요 시간: 약 2시간

print("\n" + "="*70)
print(f"  [Stage 2] Round 2 — R1 예측으로 PL 갱신")
print("="*70)

# ── R1 예측을 기존 합의에 추가 ──
# 새 합의 = (기존 합의 × 0.5) + (R1 예측 × 0.5)
# R1은 합의 PL로 학습된 것이므로 합의 PL보다 더 좋을 가능성
consensus_r2 = 0.5 * consensus_pl + 0.5 * result_r1['test_pred']

# 분산 재계산 (R1 예측도 포함)
pred_matrix_r2 = np.column_stack(
    [preds[name] for name in preds] + [result_r1['test_pred']]
)
pred_std_r2 = pred_matrix_r2.std(axis=1)

test['pseudo_label'] = consensus_r2
test['pseudo_std']   = pred_std_r2

print(f"  R2 PL = 50% 합의(R0) + 50% R1 예측")
print(f"  R2 PL mean: {consensus_r2.mean():.4f}")
print(f"  R2 신뢰도(std) mean: {pred_std_r2.mean():.4f} (R1: {pred_std.mean():.4f})")

# 새 pseudo 데이터셋 구성
train_combined_r2, stats_r2 = build_pseudo_dataset(train, test, strategy='D')

print(f"  R2 pseudo: {stats_r2['n_pseudo']}행")
print(f"  R2 weight: mean={stats_r2['weight_mean']:.4f}")

# ── Round 2 학습 ──
result_r2 = train_gbdt_round(
    train_combined_r2, test, features_pruned, seeds, round_name="R2")

print(f"\n[Stage 2 완료] R2 OOF MAE: {result_r2['oof_mae']:.4f}")
print(f"  R1 OOF: {result_r1['oof_mae']:.4f}")
print(f"  차이 (R2 - R1): {result_r2['oof_mae'] - result_r1['oof_mae']:+.4f}")

In [ ]:
# Cell 15: [Stage 3] 최종 블렌딩 + 제출 파일 생성

print("[Stage 3] 최종 블렌딩")
print("="*60)

# ── R2 단독 ──
fname = 'submission_79_r2.csv'
fpath = os.path.join(DATA_DIR, fname)
pd.DataFrame({'ID': test['ID'], TARGET: result_r2['test_pred']}).to_csv(fpath, index=False)
print(f"  ✅ {fname} — R2 단독")

# ── R1 + R2 평균 (self-ensemble) ──
r1r2_avg = (result_r1['test_pred'] + result_r2['test_pred']) / 2
r1r2_avg = np.maximum(r1r2_avg, 0)
fname = 'submission_79_r1r2_avg.csv'
fpath = os.path.join(DATA_DIR, fname)
pd.DataFrame({'ID': test['ID'], TARGET: r1r2_avg}).to_csv(fpath, index=False)
print(f"  ✅ {fname} — R1+R2 평균 (self-ensemble)")

# ── R2 + 기존 best 블렌딩 ──
for alpha in [0.3, 0.5, 0.7]:
    blend = alpha * result_r2['test_pred'] + (1 - alpha) * pred_72
    blend = np.maximum(blend, 0)
    fname = f'submission_79_r2_b{int(alpha*10)}.csv'
    fpath = os.path.join(DATA_DIR, fname)
    pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(fpath, index=False)
    print(f"  ✅ {fname} — R2 {int(alpha*100)}% + 72_top2 {int((1-alpha)*100)}%")

# ── R1R2 평균 + 기존 best 블렌딩 ──
for alpha in [0.3, 0.5, 0.7]:
    blend = alpha * r1r2_avg + (1 - alpha) * pred_72
    blend = np.maximum(blend, 0)
    fname = f'submission_79_r1r2_b{int(alpha*10)}.csv'
    fpath = os.path.join(DATA_DIR, fname)
    pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(fpath, index=False)
    print(f"  ✅ {fname} — R1R2avg {int(alpha*100)}% + 72_top2 {int((1-alpha)*100)}%")

# ── 75_g5와 블렌딩 ──
if has_75:
    blend = 0.5 * result_r2['test_pred'] + 0.5 * pred_75
    blend = np.maximum(blend, 0)
    fname = 'submission_79_r2_75b5.csv'
    fpath = os.path.join(DATA_DIR, fname)
    pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(fpath, index=False)
    print(f"  ✅ {fname} — R2 50% + 75_g5 50%")

# ── R1, R2, 72, 75 전체 평균 (mega ensemble) ──
mega_preds = [result_r1['test_pred'], result_r2['test_pred'], pred_72]
if has_75:
    mega_preds.append(pred_75)
mega_avg = np.maximum(np.mean(mega_preds, axis=0), 0)
fname = 'submission_79_mega_avg.csv'
fpath = os.path.join(DATA_DIR, fname)
pd.DataFrame({'ID': test['ID'], TARGET: mega_avg}).to_csv(fpath, index=False)
print(f"  ✅ {fname} — Mega ensemble 균등 평균")

print("\n전체 제출 파일 생성 완료")

In [ ]:
# Cell 16: 최종 결과 요약

print("\n" + "="*70)
print("  [Exp 79: Pseudo Label 품질 고도화 — 최종 결과]")
print("="*70)

# R1 ↔ R2 상관관계
corr_r1_r2, _ = pearsonr(result_r1['test_pred'], result_r2['test_pred'])
corr_r2_72, _ = pearsonr(result_r2['test_pred'], pred_72)

print(f"\n  {'모델':<35} {'OOF MAE':>10}")
print("-"*50)
print(f"  {'Exp 70b (기존 PL 기반)':>35} {'8.6935':>10}")
print(f"  {'Exp 77 Full (70b PL 재현)':>35} {'8.6886':>10}")
print(f"  {'R1 (합의 PL + confidence)':>35} {result_r1['oof_mae']:>10.4f}")
print(f"  {'R2 (R1 PL 갱신 후 재학습)':>35} {result_r2['oof_mae']:>10.4f}")

print(f"\n  상관관계:")
print(f"    R1 ↔ 72_top2: {corr_r1_72:.4f}")
print(f"    R2 ↔ 72_top2: {corr_r2_72:.4f}")
print(f"    R1 ↔ R2:      {corr_r1_r2:.4f}")

print(f"\n  기존 최선: LB 10.12838 (Exp 75)")

print(f"\n  {'─'*55}")
print(f"  📋 제출 순서:")
print(f"  {'─'*55}")
print(f"  1순위: submission_79_r1.csv")
print(f"         (합의 PL 효과 순수 측정)")
print(f"  2순위: submission_79_r1_b5.csv")
print(f"         (R1 50% + 72_top2 50% — 안전 블렌딩)")

if result_r2['oof_mae'] <= result_r1['oof_mae']:
    print(f"  3순위: submission_79_r2.csv")
    print(f"         (R2 OOF가 R1보다 좋으므로 기대)")
    print(f"  4순위: submission_79_r1r2_avg.csv")
    print(f"         (R1+R2 self-ensemble)")
else:
    print(f"  3순위: submission_79_r1r2_avg.csv")
    print(f"         (R2가 R1보다 나쁨 → 평균으로 안전)")
    print(f"  4순위: submission_79_mega_avg.csv")
    print(f"         (전체 모델 평균 — 최대 다양성)")

print(f"\n  🔍 판단 기준:")
print(f"  ┌───────────────────────────────────────────────────────────┐")
print(f"  │ R1 LB < 10.128                                         │")
print(f"  │   → 합의 PL + confidence weight 효과 확인!             │")
print(f"  │   → R2 진행, 이후 R3까지 iterative refinement          │")
print(f"  │   → 최종 예측 + TabNet 블렌딩으로 추가 개선            │")
print(f"  │                                                         │")
print(f"  │ R1 LB ≈ 10.128                                         │")
print(f"  │   → PL 품질 차이 미미. 블렌딩(b5)에서 미세 개선 가능   │")
print(f"  │   → R1+72+75 mega ensemble 탐색                        │")
print(f"  │                                                         │")
print(f"  │ R1 LB > 10.129                                         │")
print(f"  │   → 합의 PL이 오히려 noise 추가. 전략 전환 필요        │")
print(f"  │   → post-processing 탐색 또는 피처 재설계              │")
print(f"  └───────────────────────────────────────────────────────────┘")
print("="*70)

## [Exp 82 사전조건] Feature Importance 분석

`avg_rank` 변수 생성. 다음 셀에서 하위 피처 제거에 사용.

다음 셀이 실행되려면 **Exp 79 R2까지 학습이 완료되어 있어야** 합니다 (`result_r2`, `train_combined`, `features_pruned` 변수 필요).

### 분석 방법
- R2 학습에 사용된 train_combined를 그대로 사용
- 마지막 fold(F5)만 재학습하여 LGB/XGB/CAT 3종의 importance 추출 (시간 절약)
- 각 모델 내에서 rank → 3개 평균 (avg_rank) → 상위/하위 식별
- avg_rank 하위 150개를 다음 셀에서 제거

> 주의: TE 피처(`layout_mean`, `layout_std`, `layout_median`, `layout_count`)는 절대 제거하지 않음 (CV 루프 내에서 동적으로 추가됨).


In [19]:
# Cell: Feature Importance 분석 — avg_rank 생성
# Exp 79 R2와 동일한 train_combined로 마지막 fold만 학습하여 importance 추출
# → avg_rank를 만들어 Exp 82 Cell 1의 하위 피처 제거에 사용

print("=" * 60)
print("[Exp 82 사전조건] Feature Importance 분석")
print("=" * 60)

# 사전조건 체크 (Exp 79 R2 셀들이 실행되어 있어야 함)
assert 'train_combined' in dir(), "train_combined 없음 — Exp 79 R2까지 셀을 먼저 실행하세요"
assert 'features_pruned' in dir(), "features_pruned 없음 — Cell 7을 먼저 실행하세요"
assert 'result_r2' in dir(), "result_r2 없음 — Exp 79 Stage 2(Cell 14)까지 실행하세요"

# train_combined_r2를 사용 (R2 학습에 쓰인 데이터)
imp_train = train_combined_r2 if 'train_combined_r2' in dir() else train_combined

# 마지막 fold만 학습
imp_seed = 42
imp_fold_target = 4   # F5 (0-indexed)

print(f"\n  분석 대상: {len(imp_train)}행, {len(features_pruned)}개 피처")
print(f"  학습 설정: seed={imp_seed}, fold=F{imp_fold_target+1} only")

imp_lgb_dict = {}
imp_xgb_dict = {}
imp_cat_dict = {}

for fold, (tr_idx, val_idx) in enumerate(
        gkf.split(imp_train, imp_train[TARGET],
                  groups=imp_train['layout_id'])):
    if fold != imp_fold_target:
        continue

    X_tr = imp_train.iloc[tr_idx].copy()
    y_tr = imp_train.iloc[tr_idx][TARGET].values
    sw   = make_sample_weight_v2(imp_train.iloc[tr_idx])
    X_val = imp_train.iloc[val_idx].copy()
    y_val = imp_train.iloc[val_idx][TARGET].values

    # CV-safe TE
    X_tr, X_val, te_cols = apply_smoothed_te(X_tr, X_val, TARGET, k=30)
    X_tr.fillna(0, inplace=True)
    X_val.fillna(0, inplace=True)

    feats = features_pruned + te_cols
    y_tr_t  = target_forward(y_tr)
    y_val_t = target_forward(y_val)

    # ── LGB ──
    print("\n  LGB 학습 중...")
    m_lgb = LGBMRegressor(**{**LGB_PARAMS, 'random_state': imp_seed})
    m_lgb.fit(X_tr[feats], y_tr_t, sample_weight=sw,
              eval_set=[(X_val[feats], y_val_t)],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
    for f, imp in zip(feats, m_lgb.feature_importances_):
        imp_lgb_dict[f] = imp
    del m_lgb; gc.collect()

    # ── XGB ──
    print("  XGB 학습 중...")
    m_xgb = xgb.XGBRegressor(**{**XGB_PARAMS, 'random_state': imp_seed})
    m_xgb.fit(X_tr[feats], y_tr_t, sample_weight=sw,
              eval_set=[(X_val[feats], y_val_t)], verbose=False)
    for f, imp in zip(feats, m_xgb.feature_importances_):
        imp_xgb_dict[f] = imp
    del m_xgb; gc.collect()

    # ── CAT ──
    print("  CAT 학습 중...")
    m_cat = CatBoostRegressor(**{**CAT_PARAMS, 'random_seed': imp_seed})
    m_cat.fit(X_tr[feats], y_tr_t, sample_weight=sw,
              eval_set=(X_val[feats], y_val_t),
              early_stopping_rounds=100, verbose=0)
    for f, imp in zip(feats, m_cat.get_feature_importance()):
        imp_cat_dict[f] = imp
    del m_cat; gc.collect()
    break

# ── DataFrame으로 통합 ──
imp_df = pd.DataFrame({
    'feature': features_pruned,
    'lgb_imp': [imp_lgb_dict.get(f, 0) for f in features_pruned],
    'xgb_imp': [imp_xgb_dict.get(f, 0) for f in features_pruned],
    'cat_imp': [imp_cat_dict.get(f, 0) for f in features_pruned],
})

# 각 모델 내 rank (1 = 최상위, N = 최하위)
imp_df['lgb_rank'] = imp_df['lgb_imp'].rank(ascending=False, method='min')
imp_df['xgb_rank'] = imp_df['xgb_imp'].rank(ascending=False, method='min')
imp_df['cat_rank'] = imp_df['cat_imp'].rank(ascending=False, method='min')
imp_df['avg_rank_val'] = imp_df[['lgb_rank', 'xgb_rank', 'cat_rank']].mean(axis=1)

imp_df = imp_df.sort_values('avg_rank_val').reset_index(drop=True)

# avg_rank: feature -> rank Series (작은 값=상위, 큰 값=하위)
# Exp 82 Cell 1에서 avg_rank.tail(150).index로 사용
avg_rank = pd.Series(
    imp_df['avg_rank_val'].values,
    index=imp_df['feature'].values
).sort_values()

print(f"\n  분석 완료: {len(avg_rank)}개 피처")
print(f"\n  [상위 15개 — 가장 중요한 피처]")
for i, (feat, r) in enumerate(avg_rank.head(15).items(), 1):
    print(f"    {i:2d}. {feat:50s}  avg_rank={r:.1f}")

print(f"\n  [하위 15개 — 가장 덜 중요한 피처 (제거 후보)]")
for i, (feat, r) in enumerate(avg_rank.tail(15).items(), 1):
    print(f"    {i:2d}. {feat:50s}  avg_rank={r:.1f}")

# lag24 / lag20 / position_in_range 비율 확인
lag24_count = sum(1 for f in avg_rank.tail(150).index if 'lag24' in f)
lag20_count = sum(1 for f in avg_rank.tail(150).index if 'lag20' in f)
pir_count   = sum(1 for f in avg_rank.tail(150).index if 'position_in_range' in f)
print(f"\n  하위 150개 중:")
print(f"    lag24 계열: {lag24_count}개")
print(f"    lag20 계열: {lag20_count}개")
print(f"    position_in_range 계열: {pir_count}개")

print(f"\n  → 다음 셀에서 avg_rank.tail(150) 제거")


[Exp 82 사전조건] Feature Importance 분석

  분석 대상: 297600행, 446개 피처
  학습 설정: seed=42, fold=F5 only

  LGB 학습 중...
  XGB 학습 중...
  CAT 학습 중...


Default metric period is 5 because MAE is/are not implemented for GPU



  분석 완료: 446개 피처

  [상위 15개 — 가장 중요한 피처]
     1. pack_station_pressure                               avg_rank=5.0
     2. pack_utilization_roll20_mean                        avg_rank=5.7
     3. avg_trip_distance                                   avg_rank=10.7
     4. max_zone_density_roll20_mean                        avg_rank=12.3
     5. layout_compactness                                  avg_rank=13.3
     6. congestion_score_exp_mean                           avg_rank=14.0
     7. zone_dispersion                                     avg_rank=18.3
     8. robot_active_vs_cummin                              avg_rank=19.0
     9. pack_utilization_roll14_mean                        avg_rank=20.7
    10. max_zone_density                                    avg_rank=20.7
    11. intersection_count_x_congestion_score               avg_rank=22.0
    12. task_intensity                                      avg_rank=25.7
    13. pack_utilization_roll10_mean                        avg_rank=27.

## [Exp 82] 피처 재설계 — 본격 실행

다음 3개 셀은 `exp82_cells.py`의 내용입니다.

### 사전조건
- Cells 1~16(Exp 79 R2까지)이 모두 실행되어 있어야 함
- 직전 셀에서 `avg_rank`가 생성되어 있어야 함
- `pred_r2_test`는 `result_r2['test_pred']`로 alias됨 (다음 Cell 1 끝부분에서 정의)

### 셀 구성
| 셀 | 내용 | 소요 시간 |
|----|------|----------|
| Cell 1 | 피처 제거 + 신규 피처 생성 (features_v2) | ~5분 |
| Cell 2 | 1-seed 빠른 비교 (ORIG vs V2) | ~30분 |
| Cell 3 | V2 10-seed 학습 + 제출 파일 | ~2시간 |

> Cell 2 결과가 명백히 나쁘면 (V2 OOF − ORIG OOF > +0.005) Cell 3은 스킵 권장.


In [20]:
# Cell 1: 피처 제거 + 신규 피처 정의

print("=" * 60)
print("[Exp 82] 피처 재설계")
print("=" * 60)

# ── ① 하위 피처 제거 리스트 (importance 분석 기반) ──
# avg_rank 하위 150개 제거
# (하위 50은 대부분 zero, 50~150은 importance 1~3 수준)
remove_features = set(avg_rank.tail(150).index)

# TE 피처는 절대 제거하지 않음 (CV 루프 내에서 추가되므로)
remove_features -= {'layout_mean', 'layout_std', 'layout_median', 'layout_count'}

features_slim = [f for f in features_pruned if f not in remove_features]
print(f"  기존 피처: {len(features_pruned)}개")
print(f"  제거: {len(remove_features)}개")
print(f"  남은 피처: {len(features_slim)}개")

# ── ② 신규 피처 생성 ──
print(f"\n[신규 피처 생성]")
new_feats = []

# --- A. 2차 차분 (acceleration) ---
# 핵심 시계열의 "변화의 변화" — 추세 가속/감속 감지
ACCEL_COLS = ['congestion_score', 'order_inflow_15m', 'robot_active',
              'pack_utilization', 'max_zone_density', 'battery_mean']

for col in ACCEL_COLS:
    if col not in train_combined.columns:
        continue
    fname = f'{col}_accel'
    # 1차 차분의 차분 = 2차 차분
    for df in [train_combined, test]:
        vel = df.groupby('scenario_id')[col].diff(1)
        df[fname] = vel.groupby(df['scenario_id']).diff(1)
    new_feats.append(fname)

print(f"  A. 2차 차분 (acceleration): {len([f for f in new_feats if 'accel' in f])}개")

# --- B. 핵심 피처 간 비율/차이 ---
# 상위 importance 피처끼리의 교호작용
RATIO_PAIRS = [
    ('congestion_score', 'pack_utilization'),
    ('congestion_score', 'max_zone_density'),
    ('order_inflow_15m', 'robot_active'),
    ('order_inflow_15m', 'pack_utilization'),
    ('robot_active', 'max_zone_density'),
    ('battery_mean', 'congestion_score'),
    ('loading_dock_util', 'staging_area_util'),
    ('congestion_score', 'robot_idle'),
]

for c1, c2 in RATIO_PAIRS:
    if c1 in train_combined.columns and c2 in train_combined.columns:
        fname_r = f'new_{c1}_div_{c2}'
        fname_d = f'new_{c1}_minus_{c2}'
        for df in [train_combined, test]:
            df[fname_r] = df[c1] / (df[c2] + 1e-6)
            df[fname_d] = df[c1] - df[c2]
        new_feats.extend([fname_r, fname_d])

print(f"  B. 핵심 피처 비율/차이: {len([f for f in new_feats if 'div' in f or 'minus' in f])}개")

# --- C. 지수 가중 이동평균 (EWM) ---
# rolling mean과 다른 패턴: 최근 값에 더 높은 가중치
EWM_COLS = ['congestion_score', 'order_inflow_15m', 'robot_active',
            'pack_utilization', 'max_zone_density']

for col in EWM_COLS:
    if col not in train_combined.columns:
        continue
    for span in [5, 10]:
        fname = f'{col}_ewm{span}'
        for df in [train_combined, test]:
            # shift(1)로 현재 값 제외 (누수 방지)
            shifted = df.groupby('scenario_id')[col].shift(1)
            df[fname] = shifted.groupby(df['scenario_id']).transform(
                lambda x: x.ewm(span=span, min_periods=1).mean()
            )
        new_feats.append(fname)

print(f"  C. 지수 가중 이동평균 (EWM): {len([f for f in new_feats if 'ewm' in f])}개")

# --- D. 현재값 vs EWM 비율 (추세 이탈 감지) ---
for col in EWM_COLS:
    if col not in train_combined.columns:
        continue
    ewm_col = f'{col}_ewm10'
    if ewm_col in train_combined.columns:
        fname = f'{col}_vs_ewm10'
        for df in [train_combined, test]:
            df[fname] = df[col] / (df[ewm_col] + 1e-6)
        new_feats.append(fname)

print(f"  D. 현재값 vs EWM 비율: {len([f for f in new_feats if 'vs_ewm' in f])}개")

# --- E. Scenario 내 순위 (percentile) ---
# 현재 시점이 scenario 내에서 어느 수준인지
RANK_COLS = ['congestion_score', 'order_inflow_15m', 'pack_utilization']

for col in RANK_COLS:
    if col not in train_combined.columns:
        continue
    fname = f'{col}_scen_pctrank'
    for df in [train_combined, test]:
        df[fname] = df.groupby('scenario_id')[col].rank(pct=True)
    new_feats.append(fname)

print(f"  E. Scenario 내 순위: {len([f for f in new_feats if 'pctrank' in f])}개")

# --- F. 핵심 피처의 rolling 변동계수 (CV = std/mean) ---
CV_COLS = ['congestion_score', 'order_inflow_15m', 'robot_active']

for col in CV_COLS:
    if col not in train_combined.columns:
        continue
    fname = f'{col}_roll10_cv'
    for df in [train_combined, test]:
        shifted = df.groupby('scenario_id')[col].shift(1)
        grp = shifted.groupby(df['scenario_id'])
        roll_mean = grp.rolling(10, min_periods=1).mean().reset_index(level=0, drop=True)
        roll_std = grp.rolling(10, min_periods=1).std().reset_index(level=0, drop=True)
        df[fname] = roll_std / (roll_mean + 1e-6)
    new_feats.append(fname)

print(f"  F. Rolling 변동계수: {len([f for f in new_feats if '_cv' in f])}개")

# ── NaN 처리 ──
train_combined[new_feats] = train_combined[new_feats].fillna(0)
test[new_feats] = test[new_feats].fillna(0)

# ── 최종 피처 세트 ──
features_v2 = features_slim + new_feats
print(f"\n  신규 피처 합계: {len(new_feats)}개")
print(f"  최종 피처 세트: {len(features_v2)}개 (기존 {len(features_slim)} + 신규 {len(new_feats)})")
print(f"  기존 대비: {len(features_v2) - len(features_pruned):+d}개")

# ── Exp 79 R2 결과를 alias 변수로 노출 (Exp 82 호환) ──
# exp82_cells.py의 후속 셀에서 pred_r2_test로 참조하므로 매핑
pred_r2_test = result_r2['test_pred']


[Exp 82] 피처 재설계
  기존 피처: 446개
  제거: 150개
  남은 피처: 296개

[신규 피처 생성]
  A. 2차 차분 (acceleration): 6개
  B. 핵심 피처 비율/차이: 16개
  C. 지수 가중 이동평균 (EWM): 10개
  D. 현재값 vs EWM 비율: 5개
  E. Scenario 내 순위: 3개
  F. Rolling 변동계수: 3개

  신규 피처 합계: 43개
  최종 피처 세트: 339개 (기존 296 + 신규 43)
  기존 대비: -107개


In [ ]:
# Cell 2: 1-seed 빠른 비교 (기존 vs v2 피처)

print("=" * 60)
print("[Stage 1] 1-seed 빠른 비교: 기존(446) vs v2 피처")
print("=" * 60)

def quick_eval_1seed(tc, test_df, feat_list, label="model", seed=42):
    """1-seed 5-fold OOF MAE + test 예측"""
    om = ~tc['is_pseudo'].fillna(False).values
    y_o = tc[TARGET].values[om]

    oof_s = {k: np.zeros(len(tc)) for k in ['lgb','xgb','cat']}
    test_s = {k: np.zeros(len(test_df)) for k in ['lgb','xgb','cat']}

    for fold, (tr_idx, val_idx) in enumerate(
            gkf.split(tc, tc[TARGET], groups=tc['layout_id'])):
        X_tr = tc.iloc[tr_idx].copy()
        y_tr = tc.iloc[tr_idx][TARGET].values
        sw = make_sample_weight_v2(tc.iloc[tr_idx])
        X_val = tc.iloc[val_idx].copy()
        y_val = tc.iloc[val_idx][TARGET].values

        X_tr, X_val, te_cols = apply_smoothed_te(X_tr, X_val, TARGET, k=30)
        _, X_te, _ = apply_smoothed_te(X_tr, test_df.copy(), TARGET, k=30)
        X_tr.fillna(0, inplace=True); X_val.fillna(0, inplace=True); X_te.fillna(0, inplace=True)

        feats = feat_list + te_cols
        y_tr_t = target_forward(y_tr)
        y_val_t = target_forward(y_val)

        # LGB
        m = LGBMRegressor(**{**LGB_PARAMS, 'random_state': seed})
        m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
              eval_set=[(X_val[feats], y_val_t)],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
        oof_s['lgb'][val_idx] = np.maximum(target_inverse(m.predict(X_val[feats])), 0)
        test_s['lgb'] += np.maximum(target_inverse(m.predict(X_te[feats])), 0) / 5
        del m

        # XGB
        m = xgb.XGBRegressor(**{**XGB_PARAMS, 'random_state': seed})
        m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
              eval_set=[(X_val[feats], y_val_t)], verbose=False)
        oof_s['xgb'][val_idx] = np.maximum(target_inverse(m.predict(X_val[feats])), 0)
        test_s['xgb'] += np.maximum(target_inverse(m.predict(X_te[feats])), 0) / 5
        del m

        # CAT
        m = CatBoostRegressor(**{**CAT_PARAMS, 'random_seed': seed})
        m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
              eval_set=(X_val[feats], y_val_t), early_stopping_rounds=100, verbose=0)
        oof_s['cat'][val_idx] = np.maximum(target_inverse(m.predict(X_val[feats])), 0)
        test_s['cat'] += np.maximum(target_inverse(m.predict(X_te[feats])), 0) / 5
        del m

        f_lgb = mean_absolute_error(y_val, oof_s['lgb'][val_idx])
        f_xgb = mean_absolute_error(y_val, oof_s['xgb'][val_idx])
        f_cat = mean_absolute_error(y_val, oof_s['cat'][val_idx])
        print(f"  {label} F{fold+1} LGB={f_lgb:.4f} XGB={f_xgb:.4f} CAT={f_cat:.4f}")
        gc.collect()

    mae_l = mean_absolute_error(y_o, oof_s['lgb'][om])
    mae_x = mean_absolute_error(y_o, oof_s['xgb'][om])
    mae_c = mean_absolute_error(y_o, oof_s['cat'][om])
    inv = np.array([1/mae_l, 1/mae_x, 1/mae_c])
    w = inv / inv.sum()
    oof_e = w[0]*oof_s['lgb'][om] + w[1]*oof_s['xgb'][om] + w[2]*oof_s['cat'][om]
    test_e = np.maximum(w[0]*test_s['lgb'] + w[1]*test_s['xgb'] + w[2]*test_s['cat'], 0)
    mae_e = mean_absolute_error(y_o, oof_e)

    print(f"  {label} → Ens OOF: {mae_e:.4f} (LGB={mae_l:.4f} XGB={mae_x:.4f} CAT={mae_c:.4f})")
    return mae_e, test_e

# 기존 피처 (비교 기준)
mae_orig, test_orig = quick_eval_1seed(
    train_combined, test, features_pruned, label="ORIG(446)")

# v2 피처
mae_v2, test_v2 = quick_eval_1seed(
    train_combined, test, features_v2, label="V2")

print(f"\n{'='*60}")
print(f"  기존(446): OOF {mae_orig:.4f}")
print(f"  V2({len(features_v2)}):  OOF {mae_v2:.4f}")
print(f"  차이:      {mae_v2 - mae_orig:+.4f}")

if mae_v2 < mae_orig:
    print(f"  ✅ V2가 {mae_orig - mae_v2:.4f} 개선! → 10-seed 진행")
elif mae_v2 < mae_orig + 0.005:
    print(f"  🟡 차이 미미. 10-seed에서 역전 가능 → 진행 권장")
else:
    print(f"  🔴 V2가 나쁨. 피처 제거 수를 줄이거나 신규 피처 조정 필요")

# R2와 상관관계
corr_v2_r2, _ = pearsonr(pred_r2_test, test_v2)
print(f"  V2 ↔ R2 상관관계: {corr_v2_r2:.4f}")

In [ ]:
# Cell 3: V2 10-seed 학습 (Stage 1 결과가 좋을 때만 실행)

print(f"\n{'='*60}")
print(f"[Stage 2] V2 피처 10-seed 학습")
print(f"  피처: {len(features_v2)}개 + TE 4개")
print("=" * 60)

result_v2 = train_gbdt_round(
    train_combined, test, features_v2, seeds, round_name="V2")

print(f"\n[Stage 2 완료] V2 OOF MAE: {result_v2['oof_mae']:.4f}")
print(f"  R2 OOF MAE: 8.6906")
print(f"  차이: {result_v2['oof_mae'] - 8.6906:+.4f}")

# ── 제출 파일 생성 ──
print(f"\n[제출 파일]")

# V2 단독
fname = 'submission_82_v2.csv'
pd.DataFrame({'ID': test['ID'], TARGET: result_v2['test_pred']}).to_csv(
    os.path.join(DATA_DIR, fname), index=False)
print(f"  ✅ {fname}")

# R2와 블렌딩
for alpha in [0.3, 0.5, 0.7]:
    blend = alpha * result_v2['test_pred'] + (1 - alpha) * pred_r2_test
    blend = np.maximum(blend, 0)
    fname = f'submission_82_r2blend_a{int(alpha*10)}.csv'
    pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(
        os.path.join(DATA_DIR, fname), index=False)
    print(f"  ✅ {fname} — V2 {int(alpha*100)}% + R2 {int((1-alpha)*100)}%")

# V2 ↔ R2 상관관계
corr_final, _ = pearsonr(result_v2['test_pred'], pred_r2_test)
print(f"\n  V2 ↔ R2 상관관계 (10-seed): {corr_final:.4f}")

print(f"\n📋 제출 순서:")
print(f"  1순위: submission_82_v2.csv (V2 단독)")
print(f"  2순위: submission_82_r2blend_a5.csv (V2 50% + R2 50%)")

# Experiment 84: V3 피처 — V2 기반 추가 재설계

**기반**: Exp 82 V2 (LB **10.0813** — 현재 PB), 339 피처

**목표**: LB 10.0813 → 10.05~10.07 돌파

## 전략

V2의 성공 패턴(`하위 제거 + 신규 추가`)을 한 단계 더 진화:

| 단계 | 피처 수 | LB 개선 |
|------|---------|---------|
| Exp 79 (Phase 6 시작) | 446 | 10.1201 |
| Exp 82 V2 (검증된 패턴) | 339 | 10.0813 (−0.039) |
| **Exp 84 V3** | **목표 ~280** | **목표 10.05~10.07** |

## 변경점 (V2 대비)

### ① 추가 하위 제거 (V2 importance 재분석)
- V2의 339개 중 하위 80~120개 추가 제거
- V2에서 살아남았지만 여전히 약한 피처들 식별
- TE 피처(`layout_*`)는 절대 제거하지 않음

### ② 신규 피처 6종 (V2와 다른 차원)
| 그룹 | 내용 | 가설 |
|------|------|------|
| G. 다중 시간 스케일 비율 | rolling3 vs rolling20 등 | 단기/장기 추세 비교 |
| H. Layout × 운영 추가 교호작용 | floor_area × congestion 등 | 구조적 capacity |
| I. EWM 확장 (span=3, 20) | V2 EWM 검증됨 → 확장 | 시계열 보강 |
| J. 추세 일관성 (sign 비율) | rolling 양수/음수 비율 | 방향성 신호 |
| K. 3차 차분 (jerk) | 가속도의 변화율 | V2 acceleration 진화 |
| L. Cross-acceleration 비율 | acc 피처 간 비율 | V2 신규 피처 교호작용 |

### ③ PL은 V2 학습 시점과 동일 유지
Exp 83 검증 결과: PL refinement는 효과 없음 + eval_set 누수 발생.
**PL은 그대로(Exp 79 R2 합의 PL), 피처에만 집중**.

## 사전조건

Exp 82 노트북을 끝까지 실행한 상태:
- `result_v2`, `features_v2` (339개) 메모리에 있음
- `train_combined_r2` (Exp 79 R2 시점 PL, 누수 없음) 메모리에 있음
- Exp 79 R2까지 학습 완료

## 실행 시간

- Importance 재분석: ~10분
- V3 피처 생성: ~5분
- 1-seed 비교 (V2 vs V3): ~30분
- 10-seed 학습: ~2시간
- **합계: ~3시간**


In [22]:
# Cell 1: V2 importance 재분석 → avg_rank_v2 생성
# 목적: V2의 339개 중 추가 제거할 하위 피처 식별

print("=" * 60)
print("[Exp 84 Step 1] V2 importance 재분석")
print("=" * 60)

# ── 사전조건 체크 ──
assert 'features_v2' in dir(), "features_v2 없음 — Exp 82 Cell 1 실행하세요"
assert 'train_combined' in dir(), "train_combined 없음 — Cell _PmoKiwvCiIZ을 실행하여 train_combined를 생성하세요"
assert 'result_v2' in dir(), "result_v2 없음 — Exp 82 학습 결과 필요"

# Exp 82에서 V2 피처가 추가된 train_combined를 사용
# train_combined_r2는 PL 누수 방지를 위해 Exp 79 R2 시점의 PL을 유지하는 용도로만 사용됨.
# V2 피처를 사용한 importance 분석에는 V2 피처가 포함된 train_combined를 사용해야 함.
imp_train = train_combined
print(f"  분석 대상: {len(imp_train)}행, {len(features_v2)}개 피처")

# 마지막 fold만 학습 (시간 절약)
imp_seed = 42
imp_fold_target = 4

print(f"  학습 설정: seed={imp_seed}, fold=F{imp_fold_target+1} only (~5분)")

imp_lgb_v2 = {}
imp_xgb_v2 = {}
imp_cat_v2 = {}

for fold, (tr_idx, val_idx) in enumerate(
        gkf.split(imp_train, imp_train[TARGET],
                  groups=imp_train['layout_id'])):
    if fold != imp_fold_target:
        continue

    X_tr = imp_train.iloc[tr_idx].copy()
    y_tr = imp_train.iloc[tr_idx][TARGET].values
    sw   = make_sample_weight_v2(imp_train.iloc[tr_idx])
    X_val = imp_train.iloc[val_idx].copy()
    y_val = imp_train.iloc[val_idx][TARGET].values

    X_tr, X_val, te_cols = apply_smoothed_te(X_tr, X_val, TARGET, k=30)
    X_tr.fillna(0, inplace=True)
    X_val.fillna(0, inplace=True)

    feats = features_v2 + te_cols
    y_tr_t  = target_forward(y_tr)
    y_val_t = target_forward(y_val)

    print("  LGB 학습...")
    m = LGBMRegressor(**{**LGB_PARAMS, 'random_state': imp_seed})
    m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
          eval_set=[(X_val[feats], y_val_t)],
          callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
    for f, imp in zip(feats, m.feature_importances_):
        imp_lgb_v2[f] = imp
    del m; gc.collect()

    print("  XGB 학습...")
    m = xgb.XGBRegressor(**{**XGB_PARAMS, 'random_state': imp_seed})
    m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
          eval_set=[(X_val[feats], y_val_t)], verbose=False)
    for f, imp in zip(feats, m.feature_importances_):
        imp_xgb_v2[f] = imp
    del m; gc.collect()

    print("  CAT 학습...")
    m = CatBoostRegressor(**{**CAT_PARAMS, 'random_seed': imp_seed})
    m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
          eval_set=(X_val[feats], y_val_t),
          early_stopping_rounds=100, verbose=0)
    for f, imp in zip(feats, m.get_feature_importance()):
        imp_cat_v2[f] = imp
    del m; gc.collect()
    break

# ── DataFrame 통합 ──
imp_v2_df = pd.DataFrame({
    'feature': features_v2,
    'lgb_imp': [imp_lgb_v2.get(f, 0) for f in features_v2],
    'xgb_imp': [imp_xgb_v2.get(f, 0) for f in features_v2],
    'cat_imp': [imp_cat_v2.get(f, 0) for f in features_v2],
})

imp_v2_df['lgb_rank'] = imp_v2_df['lgb_imp'].rank(ascending=False, method='min')
imp_v2_df['xgb_rank'] = imp_v2_df['xgb_imp'].rank(ascending=False, method='min')
imp_v2_df['cat_rank'] = imp_v2_df['cat_imp'].rank(ascending=False, method='min')
imp_v2_df['avg_rank'] = imp_v2_df[['lgb_rank', 'xgb_rank', 'cat_rank']].mean(axis=1)
imp_v2_df = imp_v2_df.sort_values('avg_rank').reset_index(drop=True)

avg_rank_v2 = pd.Series(
    imp_v2_df['avg_rank'].values,
    index=imp_v2_df['feature'].values
).sort_values()

print(f"\n  분석 완료: {len(avg_rank_v2)}개 피처")

# ── V2 신규 피처(43개)의 importance 분석 ──
v2_new_feats = [f for f in features_v2 if any(s in f for s in
                ['_accel', 'new_', '_ewm', '_vs_ewm', '_pctrank', '_cv'])]
print(f"\n  [V2 신규 피처 {len(v2_new_feats)}개의 rank 분포]")
new_ranks = avg_rank_v2[v2_new_feats]
print(f"    상위 25% 진입: {(new_ranks <= len(features_v2)*0.25).sum()}개")
print(f"    중위 25~75%: {((new_ranks > len(features_v2)*0.25) & (new_ranks <= len(features_v2)*0.75)).sum()}개")
print(f"    하위 25%: {(new_ranks > len(features_v2)*0.75).sum()}개")
print(f"\n  [V2 신규 피처 상위 10]")
new_sorted = avg_rank_v2[v2_new_feats].sort_values()
for i, (feat, r) in enumerate(new_sorted.head(10).items(), 1):
    print(f"    {i:2d}. {feat:50s}  rank={r:.1f}")
print(f"\n  [V2 신규 피처 하위 10]")
for i, (feat, r) in enumerate(new_sorted.tail(10).items(), 1):
    print(f"    {i:2d}. {feat:50s}  rank={r:.1f}")

# ── 추가 제거 후보 (다음 셀에서 사용) ──
print(f"\n  [전체 하위 15 — 추가 제거 후보]")
for i, (feat, r) in enumerate(avg_rank_v2.tail(15).items(), 1):
    is_new = '★' if feat in v2_new_feats else ' '
    print(f"    {i:2d}.{is_new} {feat:50s}  rank={r:.1f}")

[Exp 84 Step 1] V2 importance 재분석
  분석 대상: 297600행, 339개 피처
  학습 설정: seed=42, fold=F5 only (~5분)
  LGB 학습...
  XGB 학습...
  CAT 학습...


Default metric period is 5 because MAE is/are not implemented for GPU



  분석 완료: 339개 피처

  [V2 신규 피처 43개의 rank 분포]
    상위 25% 진입: 7개
    중위 25~75%: 26개
    하위 25%: 10개

  [V2 신규 피처 상위 10]
     1. congestion_score_scen_pctrank                       rank=13.7
     2. robot_active_roll10_cv                              rank=36.0
     3. pack_utilization_scen_pctrank                       rank=52.3
     4. new_robot_active_div_max_zone_density               rank=54.3
     5. new_order_inflow_15m_div_robot_active               rank=57.3
     6. new_congestion_score_minus_robot_idle               rank=58.7
     7. new_congestion_score_div_pack_utilization           rank=70.7
     8. new_congestion_score_minus_pack_utilization         rank=86.0
     9. max_zone_density_ewm10                              rank=90.0
    10. max_zone_density_ewm5                               rank=90.7

  [V2 신규 피처 하위 10]
     1. new_congestion_score_div_robot_idle                 rank=258.7
     2. new_loading_dock_util_minus_staging_area_util       rank=297.7
     3. congestion_s

In [23]:
# Cell 2: V3 피처 정의 — 추가 제거 + 신규 6종

print("=" * 60)
print("[Exp 84 Step 2] V3 피처 정의")
print("=" * 60)

# ── ① 추가 하위 제거 ──
# V2 importance 하위 100개 제거 (조정 가능)
N_REMOVE_V3 = 100
remove_v3 = set(avg_rank_v2.tail(N_REMOVE_V3).index)

# TE 피처 보호 (혹시 들어와도 제외)
remove_v3 -= {'layout_mean', 'layout_std', 'layout_median', 'layout_count'}

features_v3_slim = [f for f in features_v2 if f not in remove_v3]
print(f"  V2 피처: {len(features_v2)}개")
print(f"  추가 제거: {len(remove_v3)}개 (V2 하위 {N_REMOVE_V3}개)")
print(f"  남은 피처: {len(features_v3_slim)}개")

# ── ② V3 신규 피처 생성 ──
print(f"\n[V3 신규 피처 생성]")
v3_new_feats = []

# 컬럼 가용성 체크 헬퍼
def col_exists(df, col):
    return col in df.columns

# --- G. 다중 시간 스케일 비율 (rolling3 vs rolling20) ---
# 단기 추세와 장기 추세 비교 — 추세 가속/감속 신호
SCALE_RATIO_COLS = ['congestion_score', 'order_inflow_15m', 'robot_active',
                    'pack_utilization', 'max_zone_density']

for col in SCALE_RATIO_COLS:
    short_col = f'{col}_roll3_mean'
    long_col  = f'{col}_roll20_mean'
    if col_exists(train_combined, short_col) and col_exists(train_combined, long_col):
        fname = f'v3_{col}_short_long_ratio'
        for df in [train_combined, test]:
            df[fname] = df[short_col] / (df[long_col] + 1e-6)
        v3_new_feats.append(fname)

n_g = sum(1 for f in v3_new_feats if 'short_long' in f)
print(f"  G. 다중 시간 스케일 비율: {n_g}개")

# --- H. Layout × 운영 추가 교호작용 ---
# 기존엔 aisle_width × congestion 등이 있었지만, 추가 조합 시도
LAYOUT_COLS = ['floor_area_sqm', 'pack_station_count', 'charger_count']
OPS_COLS    = ['congestion_score', 'pack_utilization', 'order_inflow_15m']

for l in LAYOUT_COLS:
    for o in OPS_COLS:
        if col_exists(train_combined, l) and col_exists(train_combined, o):
            fname = f'v3_{l}_x_{o}'
            for df in [train_combined, test]:
                df[fname] = df[l] * df[o]
            v3_new_feats.append(fname)
            # 비율도
            fname_r = f'v3_{o}_per_{l}'
            for df in [train_combined, test]:
                df[fname_r] = df[o] / (df[l] + 1e-6)
            v3_new_feats.append(fname_r)

n_h = sum(1 for f in v3_new_feats if f.startswith('v3_') and ('_x_' in f or '_per_' in f))
print(f"  H. Layout × 운영 교호작용: {n_h}개")

# --- I. EWM 확장 (span=3, span=20) ---
# V2의 EWM(5,10) 효과 검증됨 → 더 짧은(3)과 긴(20) span 추가
EWM_EXT_COLS = ['congestion_score', 'order_inflow_15m', 'robot_active',
                'pack_utilization', 'max_zone_density']

for col in EWM_EXT_COLS:
    if not col_exists(train_combined, col):
        continue
    for span in [3, 20]:
        fname = f'v3_{col}_ewm{span}'
        for df in [train_combined, test]:
            shifted = df.groupby('scenario_id')[col].shift(1)
            df[fname] = shifted.groupby(df['scenario_id']).transform(
                lambda x: x.ewm(span=span, min_periods=1).mean()
            )
        v3_new_feats.append(fname)

n_i = sum(1 for f in v3_new_feats if '_ewm' in f and f.startswith('v3_'))
print(f"  I. EWM 확장 (span 3, 20): {n_i}개")

# --- J. 추세 일관성 (sign 비율) ---
# rolling window 안에서 양의 변화 비율 — 추세 방향 안정성
TREND_COLS = ['congestion_score', 'order_inflow_15m', 'robot_active', 'pack_utilization']

for col in TREND_COLS:
    if not col_exists(train_combined, col):
        continue
    fname = f'v3_{col}_pos_ratio_10'
    for df in [train_combined, test]:
        diff = df.groupby('scenario_id')[col].diff(1)
        # rolling sign positive ratio
        sign_pos = (diff > 0).astype(float)
        shifted_sign = sign_pos.groupby(df['scenario_id']).shift(1)
        df[fname] = shifted_sign.groupby(df['scenario_id']).transform(
            lambda x: x.rolling(10, min_periods=1).mean()
        )
    v3_new_feats.append(fname)

n_j = sum(1 for f in v3_new_feats if 'pos_ratio' in f)
print(f"  J. 추세 일관성: {n_j}개")

# --- K. 3차 차분 (jerk = acceleration의 변화) ---
# V2의 acceleration 효과 검증됨 → 한 단계 더
JERK_COLS = ['congestion_score', 'order_inflow_15m', 'pack_utilization']

for col in JERK_COLS:
    accel_col = f'{col}_accel'  # V2에서 이미 만들어진 컬럼
    if not col_exists(train_combined, accel_col):
        # V2 셀에서 만들어졌어야 함. 없으면 스킵
        continue
    fname = f'v3_{col}_jerk'
    for df in [train_combined, test]:
        df[fname] = df.groupby('scenario_id')[accel_col].diff(1)
    v3_new_feats.append(fname)

n_k = sum(1 for f in v3_new_feats if '_jerk' in f)
print(f"  K. 3차 차분 (jerk): {n_k}개")

# --- L. Cross-acceleration 비율 ---
# V2 acceleration 피처들 간의 비율 (서로 다른 속도로 변하는 영역 식별)
CROSS_ACC_PAIRS = [
    ('congestion_score_accel', 'order_inflow_15m_accel'),
    ('congestion_score_accel', 'pack_utilization_accel'),
    ('robot_active_accel', 'order_inflow_15m_accel'),
]

for c1, c2 in CROSS_ACC_PAIRS:
    if col_exists(train_combined, c1) and col_exists(train_combined, c2):
        fname = f'v3_{c1}_div_{c2}'.replace('_accel', '_a')  # 이름 짧게
        for df in [train_combined, test]:
            df[fname] = df[c1] / (df[c2].abs() + 1e-6)
        v3_new_feats.append(fname)

n_l = sum(1 for f in v3_new_feats if 'div' in f and f.startswith('v3_'))
print(f"  L. Cross-acceleration 비율: {n_l}개")

# ── NaN 처리 ──
train_combined[v3_new_feats] = train_combined[v3_new_feats].fillna(0)
test[v3_new_feats] = test[v3_new_feats].fillna(0)

# train_combined_r2도 갱신 (Exp 79 R2 시점 train_combined)
if 'train_combined_r2' in dir():
    # train_combined_r2는 train_combined를 기반으로 만들어진 거라 같은 컬럼 가짐
    # 단, 인덱스가 다를 수 있어서 재구성 필요
    # 더 안전하게: train_combined_r2를 다시 build_pseudo_dataset로 만듦
    # 근데 이미 메모리에 있는 train_combined가 새 컬럼을 가졌으면 OK
    # 확인:
    if v3_new_feats[0] in train_combined_r2.columns:
        print(f"  ✓ train_combined_r2도 새 컬럼 자동 보유")
    else:
        # 재구성 필요
        print(f"  ⚠ train_combined_r2 재구성 필요...")
        # 간단 처리: 같은 행 순서면 column merge
        for f in v3_new_feats:
            if f not in train_combined_r2.columns:
                # train_combined의 비-pseudo 행에서 가져옴
                # 안전한 방법: build_pseudo_dataset 재호출
                pass
        # 가장 안전: 재호출
        # 근데 consensus_pl, pred_std 변경 안 했으니 결과 동일
        train_combined_r2_new, _ = build_pseudo_dataset(train, test, strategy='D')
        train_combined_r2 = train_combined_r2_new
        print(f"  ✓ train_combined_r2 재구성 완료")

# ── 최종 V3 피처 세트 ──
features_v3 = features_v3_slim + v3_new_feats
print(f"\n  V3 신규 피처 합계: {len(v3_new_feats)}개")
print(f"  최종 V3 피처 세트: {len(features_v3)}개")
print(f"    = V2 {len(features_v2)} - 제거 {len(remove_v3)} + 신규 {len(v3_new_feats)}")
print(f"  V2 대비: {len(features_v3) - len(features_v2):+d}개")


[Exp 84 Step 2] V3 피처 정의
  V2 피처: 339개
  추가 제거: 100개 (V2 하위 100개)
  남은 피처: 239개

[V3 신규 피처 생성]
  G. 다중 시간 스케일 비율: 0개
  H. Layout × 운영 교호작용: 18개
  I. EWM 확장 (span 3, 20): 10개
  J. 추세 일관성: 4개
  K. 3차 차분 (jerk): 3개
  L. Cross-acceleration 비율: 3개
  ⚠ train_combined_r2 재구성 필요...
  ✓ train_combined_r2 재구성 완료

  V3 신규 피처 합계: 38개
  최종 V3 피처 세트: 277개
    = V2 339 - 제거 100 + 신규 38
  V2 대비: -62개


In [ ]:
# Cell 3: 1-seed 빠른 비교 — V2 vs V3

print("=" * 60)
print("[Exp 84 Step 3] 1-seed 비교: V2(8.6438) vs V3")
print("=" * 60)

# train_combined_r2가 있으면 그걸 사용 (Exp 79 R2 시점 PL)
imp_train = train_combined_r2 if 'train_combined_r2' in dir() else train_combined

# Exp 82 quick_eval_1seed 함수가 메모리에 있는지 확인
if 'quick_eval_1seed' not in dir():
    print("  quick_eval_1seed 함수 정의...")
    def quick_eval_1seed(tc, test_df, feat_list, label="model", seed=42):
        om = ~tc['is_pseudo'].fillna(False).values
        y_o = tc[TARGET].values[om]
        oof_s = {k: np.zeros(len(tc)) for k in ['lgb','xgb','cat']}
        test_s = {k: np.zeros(len(test_df)) for k in ['lgb','xgb','cat']}
        for fold, (tr_idx, val_idx) in enumerate(
                gkf.split(tc, tc[TARGET], groups=tc['layout_id'])):
            X_tr = tc.iloc[tr_idx].copy()
            y_tr = tc.iloc[tr_idx][TARGET].values
            sw = make_sample_weight_v2(tc.iloc[tr_idx])
            X_val = tc.iloc[val_idx].copy()
            y_val = tc.iloc[val_idx][TARGET].values
            X_tr, X_val, te_cols = apply_smoothed_te(X_tr, X_val, TARGET, k=30)
            _, X_te, _ = apply_smoothed_te(X_tr, test_df.copy(), TARGET, k=30)
            X_tr.fillna(0, inplace=True); X_val.fillna(0, inplace=True); X_te.fillna(0, inplace=True)
            feats = feat_list + te_cols
            y_tr_t = target_forward(y_tr); y_val_t = target_forward(y_val)
            m = LGBMRegressor(**{**LGB_PARAMS, 'random_state': seed})
            m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
                  eval_set=[(X_val[feats], y_val_t)],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
            oof_s['lgb'][val_idx] = np.maximum(target_inverse(m.predict(X_val[feats])), 0)
            test_s['lgb'] += np.maximum(target_inverse(m.predict(X_te[feats])), 0) / 5
            del m
            m = xgb.XGBRegressor(**{**XGB_PARAMS, 'random_state': seed})
            m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
                  eval_set=[(X_val[feats], y_val_t)], verbose=False)
            oof_s['xgb'][val_idx] = np.maximum(target_inverse(m.predict(X_val[feats])), 0)
            test_s['xgb'] += np.maximum(target_inverse(m.predict(X_te[feats])), 0) / 5
            del m
            m = CatBoostRegressor(**{**CAT_PARAMS, 'random_seed': seed})
            m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
                  eval_set=(X_val[feats], y_val_t), early_stopping_rounds=100, verbose=0)
            oof_s['cat'][val_idx] = np.maximum(target_inverse(m.predict(X_val[feats])), 0)
            test_s['cat'] += np.maximum(target_inverse(m.predict(X_te[feats])), 0) / 5
            del m
            f_lgb = mean_absolute_error(y_val, oof_s['lgb'][val_idx])
            f_xgb = mean_absolute_error(y_val, oof_s['xgb'][val_idx])
            f_cat = mean_absolute_error(y_val, oof_s['cat'][val_idx])
            print(f"  {label} F{fold+1} LGB={f_lgb:.4f} XGB={f_xgb:.4f} CAT={f_cat:.4f}")
            gc.collect()
        mae_l = mean_absolute_error(y_o, oof_s['lgb'][om])
        mae_x = mean_absolute_error(y_o, oof_s['xgb'][om])
        mae_c = mean_absolute_error(y_o, oof_s['cat'][om])
        inv = np.array([1/mae_l, 1/mae_x, 1/mae_c])
        w = inv / inv.sum()
        oof_e = w[0]*oof_s['lgb'][om] + w[1]*oof_s['xgb'][om] + w[2]*oof_s['cat'][om]
        test_e = np.maximum(w[0]*test_s['lgb'] + w[1]*test_s['xgb'] + w[2]*test_s['cat'], 0)
        mae_e = mean_absolute_error(y_o, oof_e)
        print(f"  {label} → Ens OOF: {mae_e:.4f} (LGB={mae_l:.4f} XGB={mae_x:.4f} CAT={mae_c:.4f})")
        return mae_e, test_e

# V3 1-seed 평가
mae_v3_1seed, test_v3_1seed = quick_eval_1seed(
    imp_train, test, features_v3, label="V3")

print(f"\n{'='*60}")
print(f"  Exp 82 V2 (10-seed): OOF 8.6438, LB 10.0813")
print(f"  Exp 84 V3 (1-seed):  OOF {mae_v3_1seed:.4f}")
print(f"  V2 1-seed (기억나는 값): OOF 8.6489 (참고용)")
print(f"\n  주의: 1-seed → 10-seed 시 보통 OOF -0.005~0.01 개선됨")
print(f"        V3 1-seed가 V2 1-seed(8.6489) 대비 어떤지 확인")

# V2와 상관관계
corr_v3_v2, _ = pearsonr(result_v2['test_pred'], test_v3_1seed)
print(f"\n  V3 ↔ V2 상관관계: {corr_v3_v2:.4f}")
print(f"    (V2 ↔ R2: 0.9992)")

# 판단
v2_1seed_ref = 8.6489
if mae_v3_1seed < v2_1seed_ref - 0.005:
    print(f"\n  ✅ V3가 V2보다 명백히 좋음 ({v2_1seed_ref - mae_v3_1seed:.4f} 개선)")
    print(f"     → 10-seed 진행 권장")
elif mae_v3_1seed < v2_1seed_ref + 0.005:
    print(f"\n  🟡 차이 미미 ({mae_v3_1seed - v2_1seed_ref:+.4f})")
    print(f"     → 10-seed에서 변동 가능. 진행해서 확인 권장")
else:
    print(f"\n  🔴 V3가 V2보다 나쁨 ({mae_v3_1seed - v2_1seed_ref:+.4f})")
    print(f"     → 신규 피처 그룹 ablation 필요. N_REMOVE_V3를 줄이거나 신규 피처 조정")
    print(f"     → 그래도 10-seed 시도해볼 가치 있음 (fold 운빨 가능성)")


In [ ]:
# Cell 4: V3 10-seed 학습 + 제출 파일

print(f"\n{'='*60}")
print(f"[Exp 84 Step 4] V3 10-seed 학습")
print(f"  피처: {len(features_v3)}개 + TE 4개")
print(f"  Train: {len(imp_train)}행 (Exp 79 R2 시점 PL — 누수 없음)")
print("=" * 60)

result_v3 = train_gbdt_round(
    imp_train, test, features_v3, seeds, round_name="V3")

print(f"\n[Exp 84 V3 학습 완료]")
print(f"  V3 OOF MAE: {result_v3['oof_mae']:.4f}")
print(f"  V2 OOF MAE: 8.6438 (LB 10.0813)")
print(f"  R2 OOF MAE: 8.6906 (LB 10.1201)")
print(f"  V3 - V2 OOF: {result_v3['oof_mae'] - 8.6438:+.4f}")

# ── 제출 파일 생성 ──
print(f"\n[제출 파일 생성]")

# V3 단독
fname = 'submission_84_v3.csv'
pd.DataFrame({'ID': test['ID'], TARGET: result_v3['test_pred']}).to_csv(
    os.path.join(DATA_DIR, fname), index=False)
print(f"  ✅ {fname} — V3 단독 (V2 대비 피처 재설계)")

# V2와 블렌딩 (안전장치)
pred_v2 = result_v2['test_pred']
for alpha in [0.3, 0.5, 0.7]:
    blend = alpha * result_v3['test_pred'] + (1 - alpha) * pred_v2
    blend = np.maximum(blend, 0)
    fname = f'submission_84_v3_v2b{int(alpha*10)}.csv'
    pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(
        os.path.join(DATA_DIR, fname), index=False)
    print(f"  ✅ {fname} — V3 {int(alpha*100)}% + V2 {int((1-alpha)*100)}%")

# V3 + 75_g5 (TabNet 다양성)
v75_path = os.path.join(DATA_DIR, 'submission_75_g5.csv')
if os.path.exists(v75_path):
    pred_75 = pd.read_csv(v75_path).set_index('ID').loc[test['ID'].values][TARGET].values
    blend = 0.9 * result_v3['test_pred'] + 0.1 * pred_75
    blend = np.maximum(blend, 0)
    fname = 'submission_84_v3_75b1.csv'
    pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(
        os.path.join(DATA_DIR, fname), index=False)
    print(f"  ✅ {fname} — V3 90% + 75_g5 10% (TabNet 다양성)")

# ── 상관관계 분석 ──
corr_v3_v2_full, _ = pearsonr(result_v3['test_pred'], pred_v2)
print(f"\n  V3 ↔ V2 상관관계 (10-seed): {corr_v3_v2_full:.4f}")

# ── 제출 우선순위 ──
print(f"\n  {'─'*55}")
print(f"  📋 제출 우선순위:")
print(f"  {'─'*55}")

if result_v3['oof_mae'] < 8.6438 - 0.005:
    # V3가 V2보다 명백히 좋음
    print(f"  1순위: submission_84_v3.csv")
    print(f"         (V3 OOF가 V2 대비 {8.6438 - result_v3['oof_mae']:.4f} 개선)")
    print(f"         예상 LB: 10.06~10.07")
    print(f"  2순위: submission_84_v3_v2b7.csv")
    print(f"         (V3 70% + V2 30% — 안전 보험)")
    print(f"  3순위: submission_84_v3_75b1.csv")
    print(f"         (V3 + TabNet 미세 다양성)")
elif result_v3['oof_mae'] < 8.6438 + 0.005:
    # V3가 V2와 비슷
    print(f"  1순위: submission_84_v3.csv")
    print(f"         (OOF 비슷, LB는 미세하게 다를 수 있음)")
    print(f"  2순위: submission_84_v3_v2b5.csv")
    print(f"         (50/50 블렌딩 — 가장 안전)")
    print(f"  3순위: submission_84_v3_v2b3.csv")
    print(f"         (V2 70% 비중 — V2가 PB라 보호)")
else:
    # V3가 V2보다 나쁨
    print(f"  ⚠️ V3 OOF가 V2(8.6438)보다 나쁨")
    print(f"  → V2 단독(LB 10.0813)이 여전히 최선일 가능성")
    print(f"  1순위: submission_84_v3_v2b3.csv")
    print(f"         (V3 30% + V2 70% — V2 보호)")
    print(f"  2순위: submission_84_v3_v2b5.csv")
    print(f"         (50/50 블렌딩)")
    print(f"  → 다음 실험: 신규 피처 ablation으로 어느 그룹이 noise인지 식별")

# ── 모델별 OOF ──
print(f"\n  [모델별 단일 OOF]")
print(f"    LGB weight: {result_v3['weights'][0]:.4f}")
print(f"    XGB weight: {result_v3['weights'][1]:.4f}")
print(f"    CAT weight: {result_v3['weights'][2]:.4f}")

print(f"\n{'='*60}")
print(f"  Exp 84 V3 완료")
print(f"{'='*60}")


# Experiment 85: V4 피처 — Percentile Rank 확장 + Acceleration 제거

**기반**: Exp 84 V3 (LB **10.0737** — 현재 PB), 277 피처

**목표**: LB 10.0737 → 10.05~10.07 돌파

## 핵심 가설: Percentile Rank 확장이 V4의 결정적 카드

V2 importance 분석에서 발견된 **압도적 신호**:

```
V2 신규 피처 43개 중 importance rank 상위:
  congestion_score_scen_pctrank   rank=10   ← 압도적 1위
  pack_utilization_scen_pctrank   rank=45   ← 3위
  (V2엔 단 3개만 있음, 모두 상위 진입)
```

**왜 percentile rank가 강력한가**:
- GBDT는 절대값으로 분기 → "scenario 내 상대 위치"는 직접 표현 못 함
- percentile rank가 *상대* 정보를 명시적으로 제공
- Unseen layout 일반화에 특히 강함 (절대값 분포가 달라도 상대 위치는 보편)

**V4의 결정적 변경**: percentile rank 3개 → **35+개로 대폭 확장**

## 변경점 (V3 → V4)

### ① V3 importance 재분석 → 추가 하위 제거 (~80~100개)

### ② Acceleration 계열 완전 제거
V2 importance 분석에서 발견:
```
V2 신규 acceleration (A그룹) 6개 — 모두 rank 303~329 (하위 25%)
V3 jerk (K그룹), cross-acc (L그룹) — acc 기반이라 동일 의심
```
→ V4에서 acceleration 계열 ~12개 모두 제거

### ③ Percentile Rank 대폭 확장 (V4의 핵심)

| 종류 | V2 | V4 | 증가 |
|------|-----|-----|------|
| Scenario 내 pctrank | 3개 | 18개 | +15 |
| Rolling 30 pctrank | 0개 | 12개 | +12 |
| Rolling 10 pctrank | 0개 | 8개 | +8 |
| **합계** | **3개** | **38개** | **+35** |

### ④ 보조 신규: 도달 후 경과 시간
- 최댓값 도달 후 경과 시간 (`_since_max_X`)
- 최솟값 도달 후 경과 시간 (`_since_min_X`)
- 시계열 사건 추적의 핵심 신호

## 사전조건

Exp 84 V3 학습 완료 + 메모리에 다음 변수 있음:
- `result_v3`, `features_v3` (277개)
- `train_combined_r2` (Exp 79 R2 PL, 누수 없음)
- `result_v2`, `features_v2`

## 실행 시간

- V3 importance 재분석: ~10분
- V4 피처 생성: ~5분
- 1-seed 비교: ~30분
- 10-seed 학습: ~2시간
- **합계: ~3시간**

## 안전장치

- PL은 Exp 79 R2 시점 그대로 (검증됨, 누수 없음)
- TE 피처(`layout_*`) 절대 제거 안 함
- 1-seed 검증 후에만 10-seed 진행


In [25]:
# Cell 1: V3 importance 재분석 → avg_rank_v3 생성

print("=" * 60)
print("[Exp 85 Step 1] V3 importance 재분석")
print("=" * 60)

# ── 사전조건 체크 ──
assert 'features_v3' in dir(), "features_v3 없음 — Exp 84 Cell 2 실행하세요"
assert 'result_v3' in dir(), "result_v3 없음 — Exp 84 Cell 4 실행하세요"

imp_train = train_combined # Changed from: train_combined_r2 if 'train_combined_r2' in dir() else train_combined
print(f"  분석 대상: {len(imp_train)}행, {len(features_v3)}개 피처")

imp_seed = 42
imp_fold_target = 4

print(f"  학습: seed={imp_seed}, fold=F{imp_fold_target+1} only")

imp_lgb_v3 = {}
imp_xgb_v3 = {}
imp_cat_v3 = {}

for fold, (tr_idx, val_idx) in enumerate(
        gkf.split(imp_train, imp_train[TARGET],
                  groups=imp_train['layout_id'])):
    if fold != imp_fold_target:
        continue

    X_tr = imp_train.iloc[tr_idx].copy()
    y_tr = imp_train.iloc[tr_idx][TARGET].values
    sw   = make_sample_weight_v2(imp_train.iloc[tr_idx])
    X_val = imp_train.iloc[val_idx].copy()
    y_val = imp_train.iloc[val_idx][TARGET].values

    X_tr, X_val, te_cols = apply_smoothed_te(X_tr, X_val, TARGET, k=30)
    X_tr.fillna(0, inplace=True)
    X_val.fillna(0, inplace=True)

    feats = features_v3 + te_cols
    y_tr_t  = target_forward(y_tr)
    y_val_t = target_forward(y_val)

    print("  LGB 학습...")
    m = LGBMRegressor(**{**LGB_PARAMS, 'random_state': imp_seed})
    m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
          eval_set=[(X_val[feats], y_val_t)],
          callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
    for f, imp in zip(feats, m.feature_importances_):
        imp_lgb_v3[f] = imp
    del m; gc.collect()

    print("  XGB 학습...")
    m = xgb.XGBRegressor(**{**XGB_PARAMS, 'random_state': imp_seed})
    m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
          eval_set=[(X_val[feats], y_val_t)], verbose=False)
    for f, imp in zip(feats, m.feature_importances_):
        imp_xgb_v3[f] = imp
    del m; gc.collect()

    print("  CAT 학습...")
    m = CatBoostRegressor(**{**CAT_PARAMS, 'random_seed': imp_seed})
    m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
          eval_set=(X_val[feats], y_val_t),
          early_stopping_rounds=100, verbose=0)
    for f, imp in zip(feats, m.get_feature_importance()):
        imp_cat_v3[f] = imp
    del m; gc.collect()
    break

# ── DataFrame 통합 ──
imp_v3_df = pd.DataFrame({
    'feature': features_v3,
    'lgb_imp': [imp_lgb_v3.get(f, 0) for f in features_v3],
    'xgb_imp': [imp_xgb_v3.get(f, 0) for f in features_v3],
    'cat_imp': [imp_cat_v3.get(f, 0) for f in features_v3],
})

imp_v3_df['lgb_rank'] = imp_v3_df['lgb_imp'].rank(ascending=False, method='min')
imp_v3_df['xgb_rank'] = imp_v3_df['xgb_imp'].rank(ascending=False, method='min')
imp_v3_df['cat_rank'] = imp_v3_df['cat_imp'].rank(ascending=False, method='min')
imp_v3_df['avg_rank'] = imp_v3_df[['lgb_rank', 'xgb_rank', 'cat_rank']].mean(axis=1)
imp_v3_df = imp_v3_df.sort_values('avg_rank').reset_index(drop=True)

avg_rank_v3 = pd.Series(
    imp_v3_df['avg_rank'].values,
    index=imp_v3_df['feature'].values
).sort_values()

print(f"\n  분석 완료: {len(avg_rank_v3)}개 피처")

# ── V3 신규 피처 분석 ──
v3_new_feats = [f for f in features_v3 if f.startswith('v3_')]
print(f"\n  [V3 신규 피처 {len(v3_new_feats)}개의 rank 분포]")
new_ranks_v3 = avg_rank_v3[v3_new_feats]
print(f"    상위 25% 진입: {(new_ranks_v3 <= len(features_v3)*0.25).sum()}개")
print(f"    중위 25~75%: {((new_ranks_v3 > len(features_v3)*0.25) & (new_ranks_v3 <= len(features_v3)*0.75)).sum()}개")
print(f"    하위 25%: {(new_ranks_v3 > len(features_v3)*0.75).sum()}개")

print(f"\n  [V3 신규 피처 상위 10]")
for i, (feat, r) in enumerate(avg_rank_v3[v3_new_feats].sort_values().head(10).items(), 1):
    print(f"    {i:2d}. {feat:55s}  rank={r:.1f}")

print(f"\n  [V3 신규 피처 하위 10]")
for i, (feat, r) in enumerate(avg_rank_v3[v3_new_feats].sort_values().tail(10).items(), 1):
    print(f"    {i:2d}. {feat:55s}  rank={r:.1f}")

# ── Acceleration 계열 importance 확인 ──
acc_feats = [f for f in features_v3 if 'accel' in f or '_jerk' in f or 'div_a_' in f]
print(f"\n  [Acceleration 계열 {len(acc_feats)}개의 rank]")
if acc_feats:
    acc_ranks = avg_rank_v3[acc_feats].sort_values()
    for feat, r in acc_ranks.items():
        marker = '🔴' if r > len(features_v3) * 0.75 else ('🟡' if r > len(features_v3) * 0.5 else '🟢')
        print(f"    {marker} {feat:55s}  rank={r:.1f}")


[Exp 85 Step 1] V3 importance 재분석
  분석 대상: 297600행, 277개 피처
  학습: seed=42, fold=F5 only
  LGB 학습...
  XGB 학습...
  CAT 학습...


Default metric period is 5 because MAE is/are not implemented for GPU



  분석 완료: 277개 피처

  [V3 신규 피처 38개의 rank 분포]
    상위 25% 진입: 2개
    중위 25~75%: 21개
    하위 25%: 15개

  [V3 신규 피처 상위 10]
     1. v3_order_inflow_15m_per_pack_station_count               rank=8.0
     2. v3_congestion_score_per_pack_station_count               rank=60.7
     3. v3_max_zone_density_ewm20                                rank=78.3
     4. v3_order_inflow_15m_ewm20                                rank=79.7
     5. v3_pack_utilization_per_pack_station_count               rank=100.3
     6. v3_pack_utilization_ewm20                                rank=103.0
     7. v3_max_zone_density_ewm3                                 rank=110.0
     8. v3_pack_station_count_x_pack_utilization                 rank=115.3
     9. v3_robot_active_ewm20                                    rank=121.0
    10. v3_charger_count_x_congestion_score                      rank=141.0

  [V3 신규 피처 하위 10]
     1. v3_pack_utilization_ewm3                                 rank=243.0
     2. v3_order_inflow_15m_ewm

In [26]:
# Cell 2: V4 피처 정의 — Percentile 확장 + Acceleration 제거

print("=" * 60)
print("[Exp 85 Step 2] V4 피처 정의")
print("=" * 60)

# ── ① 추가 제거: V3 하위 + acceleration 계열 ──

# V3 importance 하위 100개
N_REMOVE_BY_RANK = 100
remove_v4 = set(avg_rank_v3.tail(N_REMOVE_BY_RANK).index)
n_rank_remove = len(remove_v4)

# acceleration 계열 모두 제거 (V2 분석에서 하위 일관됨)
acc_pattern_feats = set()
for f in features_v3:
    if any(p in f for p in ['_accel', '_jerk']):
        acc_pattern_feats.add(f)
    # cross-accel 비율도
    if f.startswith('v3_') and 'div_' in f and '_a_' in f:
        acc_pattern_feats.add(f)

# 합집합 (이미 제거 후보에 포함된 것도 있을 수 있음)
remove_v4 |= acc_pattern_feats
n_acc_added = len(acc_pattern_feats - set(avg_rank_v3.tail(N_REMOVE_BY_RANK).index))

# TE 피처 보호
remove_v4 -= {'layout_mean', 'layout_std', 'layout_median', 'layout_count'}

features_v4_slim = [f for f in features_v3 if f not in remove_v4]
print(f"  V3 피처: {len(features_v3)}개")
print(f"  제거: {len(remove_v4)}개")
print(f"    - rank 하위 {N_REMOVE_BY_RANK}개에서: {n_rank_remove}개")
print(f"    - acceleration 추가: {n_acc_added}개")
print(f"  남은 피처: {len(features_v4_slim)}개")

# ── ② V4 신규 피처 — Percentile Rank 대폭 확장 ──
print(f"\n[V4 신규 피처 생성]")
v4_new_feats = []

def col_exists(df, col):
    return col in df.columns

# --- M. Scenario 내 Percentile Rank 확장 (V2의 3개 → 18개) ---
print(f"\n  M. Scenario percentile rank 확장")
PCTRANK_COLS = [
    # V2 기존 3개는 이미 features_v3에 있음 → 새로 만들지 않음
    # 추가 핵심 컬럼 (V2 importance 상위 + 핵심 운영 지표)
    'robot_active', 'max_zone_density', 'battery_mean',
    'pack_station_pressure', 'loading_dock_util', 'staging_area_util',
    'robot_idle', 'robot_charging', 'fault_count_15m',
    'blocked_path_15m',
    # rolling 결과에도 적용 (rolling이 noise smoothing이라 더 안정적 신호)
    'congestion_score_roll10_mean',
    'order_inflow_15m_roll10_mean',
    'pack_utilization_roll10_mean',
    'robot_active_roll10_mean',
    'max_zone_density_roll10_mean',
]

n_m = 0
for col in PCTRANK_COLS:
    if not col_exists(train_combined, col):
        continue
    fname = f'v4_{col}_scen_pct'
    for df in [train_combined, test]:
        df[fname] = df.groupby('scenario_id')[col].rank(pct=True)
    v4_new_feats.append(fname)
    n_m += 1

print(f"     생성: {n_m}개")

# --- N. Rolling 30 percentile rank ---
print(f"\n  N. Rolling 30 percentile rank")
ROLL_PCT_COLS = [
    'congestion_score', 'order_inflow_15m', 'pack_utilization',
    'robot_active', 'max_zone_density', 'battery_mean',
    'pack_station_pressure', 'loading_dock_util',
    'fault_count_15m', 'blocked_path_15m',
    'staging_area_util', 'robot_idle',
]

n_n = 0
for col in ROLL_PCT_COLS:
    if not col_exists(train_combined, col):
        continue
    fname = f'v4_{col}_roll30_pct'
    for df in [train_combined, test]:
        shifted = df.groupby('scenario_id')[col].shift(1)
        df[fname] = shifted.groupby(df['scenario_id']).transform(
            lambda x: x.rolling(30, min_periods=5).rank(pct=True).iloc[-1]
                      if len(x) >= 1 else np.nan
        )
        # 위 transform이 너무 느릴 수 있어 대안:
        # rolling rank는 percentile of last value within window
        # pandas rolling.rank()를 직접 사용
        df[fname] = shifted.groupby(df['scenario_id']).transform(
            lambda x: x.rolling(30, min_periods=5).rank(pct=True)
        )
    v4_new_feats.append(fname)
    n_n += 1

print(f"     생성: {n_n}개")

# --- O. Rolling 10 percentile rank (단기 신호) ---
print(f"\n  O. Rolling 10 percentile rank")
SHORT_ROLL_PCT_COLS = [
    'congestion_score', 'order_inflow_15m', 'pack_utilization',
    'robot_active', 'max_zone_density',
    'pack_station_pressure', 'loading_dock_util', 'fault_count_15m',
]

n_o = 0
for col in SHORT_ROLL_PCT_COLS:
    if not col_exists(train_combined, col):
        continue
    fname = f'v4_{col}_roll10_pct'
    for df in [train_combined, test]:
        shifted = df.groupby('scenario_id')[col].shift(1)
        df[fname] = shifted.groupby(df['scenario_id']).transform(
            lambda x: x.rolling(10, min_periods=3).rank(pct=True)
        )
    v4_new_feats.append(fname)
    n_o += 1

print(f"     생성: {n_o}개")

# --- P. 최댓값/최솟값 도달 후 경과 시간 ---
# vs_cummax는 비율, 이건 시간 — 다른 신호
print(f"\n  P. 최댓값/최솟값 도달 후 경과 시간")
SINCE_MAX_COLS = [
    'congestion_score', 'order_inflow_15m', 'pack_utilization',
    'robot_active', 'max_zone_density',
]

n_p = 0
for col in SINCE_MAX_COLS:
    if not col_exists(train_combined, col):
        continue

    # since_max: 현재까지 cummax가 갱신된 이후의 시간 step
    fname_max = f'v4_{col}_since_max'
    fname_min = f'v4_{col}_since_min'

    for df in [train_combined, test]:
        # cummax 갱신 시점 식별
        grp = df.groupby('scenario_id')[col]
        cummax = grp.cummax()
        cummin = grp.cummin()

        # cummax가 갱신될 때마다 카운터 리셋
        is_new_max = (df[col] == cummax).astype(int)
        is_new_min = (df[col] == cummin).astype(int)

        # 현재 행이 새 최댓값이면 0, 아니면 누적
        # group 내에서 cumcount 후 reset 트릭
        # 간단 방법: scenario 내에서 마지막 max 갱신 이후의 step 수
        def steps_since_event(s, event_mask):
            # event_mask: 1이면 이벤트 발생 (cummax 갱신)
            # 결과: 이벤트 이후 경과한 step 수 (0부터 시작)
            grp_idx = np.arange(len(s))
            event_indices = np.where(event_mask.values)[0]
            if len(event_indices) == 0:
                return pd.Series(grp_idx, index=s.index)
            # 각 위치에서 가장 가까운 이전 event index
            result = np.zeros(len(s), dtype=int)
            last_event = -1
            for i in range(len(s)):
                if event_mask.iloc[i]:
                    last_event = i
                    result[i] = 0
                elif last_event >= 0:
                    result[i] = i - last_event
                else:
                    result[i] = i  # 첫 max가 아직 없으면 처음부터
            return pd.Series(result, index=s.index)

        # scenario별로 적용
        max_steps = []
        min_steps = []
        for sid, sub in df.groupby('scenario_id'):
            ev_max = (sub[col] == sub[col].cummax())
            ev_min = (sub[col] == sub[col].cummin())
            max_steps.append(steps_since_event(sub[col], ev_max))
            min_steps.append(steps_since_event(sub[col], ev_min))
        df[fname_max] = pd.concat(max_steps).sort_index().values
        df[fname_min] = pd.concat(min_steps).sort_index().values

    v4_new_feats.extend([fname_max, fname_min])
    n_p += 2

print(f"     생성: {n_p}개")

# ── NaN 처리 ──
train_combined[v4_new_feats] = train_combined[v4_new_feats].fillna(0)
test[v4_new_feats] = test[v4_new_feats].fillna(0)

# ── train_combined_r2 컬럼 동기화 ──
if 'train_combined_r2' in dir():
    if v4_new_feats[0] not in train_combined_r2.columns:
        # 재구성
        print(f"\n  ⚠ train_combined_r2 재구성 중...")
        train_combined_r2_new, _ = build_pseudo_dataset(train, test, strategy='D')
        train_combined_r2 = train_combined_r2_new
        print(f"  ✓ 재구성 완료")
    else:
        print(f"\n  ✓ train_combined_r2 새 컬럼 자동 보유")

# ── 최종 V4 피처 세트 ──
features_v4 = features_v4_slim + v4_new_feats
print(f"\n  V4 신규 피처 합계: {len(v4_new_feats)}개")
print(f"  최종 V4 피처 세트: {len(features_v4)}개")
print(f"    = V3 {len(features_v3)} - 제거 {len(remove_v4)} + 신규 {len(v4_new_feats)}")
print(f"  V3 대비: {len(features_v4) - len(features_v3):+d}개")
print(f"  V2 대비: {len(features_v4) - len(features_v2):+d}개")
print(f"\n  [Percentile rank 총 개수]")
v2_pctrank = sum(1 for f in features_v4 if 'pctrank' in f and not f.startswith('v4_'))
v4_pctrank = sum(1 for f in features_v4 if 'pct' in f and f.startswith('v4_'))
print(f"    V2 시절 (scen_pctrank): {v2_pctrank}개")
print(f"    V4 신규 (모든 percentile): {v4_pctrank}개")
print(f"    합계: {v2_pctrank + v4_pctrank}개")


[Exp 85 Step 2] V4 피처 정의
  V3 피처: 277개
  제거: 100개
    - rank 하위 100개에서: 100개
    - acceleration 추가: 0개
  남은 피처: 177개

[V4 신규 피처 생성]

  M. Scenario percentile rank 확장
     생성: 12개

  N. Rolling 30 percentile rank
     생성: 12개

  O. Rolling 10 percentile rank
     생성: 8개

  P. 최댓값/최솟값 도달 후 경과 시간
     생성: 10개

  ⚠ train_combined_r2 재구성 중...
  ✓ 재구성 완료

  V4 신규 피처 합계: 42개
  최종 V4 피처 세트: 219개
    = V3 277 - 제거 100 + 신규 42
  V3 대비: -58개
  V2 대비: -120개

  [Percentile rank 총 개수]
    V2 시절 (scen_pctrank): 2개
    V4 신규 (모든 percentile): 32개
    합계: 34개


In [28]:
# Cell 3: 1-seed 비교 — V3 vs V4

print("=" * 60)
print("[Exp 85 Step 3] 1-seed 비교: V3(8.6392) vs V4")
print("=" * 60)

imp_train = train_combined # Corrected from: train_combined_r2 if 'train_combined_r2' in dir() else train_combined

if 'quick_eval_1seed' not in dir():
    print("  quick_eval_1seed 함수 정의...")
    def quick_eval_1seed(tc, test_df, feat_list, label="model", seed=42):
        om = ~tc['is_pseudo'].fillna(False).values
        y_o = tc[TARGET].values[om]
        oof_s = {k: np.zeros(len(tc)) for k in ['lgb','xgb','cat']}
        test_s = {k: np.zeros(len(test_df)) for k in ['lgb','xgb','cat']}
        for fold, (tr_idx, val_idx) in enumerate(
                gkf.split(tc, tc[TARGET], groups=tc['layout_id'])):
            X_tr = tc.iloc[tr_idx].copy()
            y_tr = tc.iloc[tr_idx][TARGET].values
            sw = make_sample_weight_v2(tc.iloc[tr_idx])
            X_val = tc.iloc[val_idx].copy()
            y_val = tc.iloc[val_idx][TARGET].values
            X_tr, X_val, te_cols = apply_smoothed_te(X_tr, X_val, TARGET, k=30)
            _, X_te, _ = apply_smoothed_te(X_tr, test_df.copy(), TARGET, k=30)
            X_tr.fillna(0, inplace=True); X_val.fillna(0, inplace=True); X_te.fillna(0, inplace=True)
            feats = feat_list + te_cols
            y_tr_t = target_forward(y_tr); y_val_t = target_forward(y_val)
            m = LGBMRegressor(**{**LGB_PARAMS, 'random_state': seed})
            m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
                  eval_set=[(X_val[feats], y_val_t)],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
            oof_s['lgb'][val_idx] = np.maximum(target_inverse(m.predict(X_val[feats])), 0)
            test_s['lgb'] += np.maximum(target_inverse(m.predict(X_te[feats])), 0) / 5
            del m
            m = xgb.XGBRegressor(**{**XGB_PARAMS, 'random_state': seed})
            m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
                  eval_set=[(X_val[feats], y_val_t)], verbose=False)
            oof_s['xgb'][val_idx] = np.maximum(target_inverse(m.predict(X_val[feats])), 0)
            test_s['xgb'] += np.maximum(target_inverse(m.predict(X_te[feats])), 0) / 5
            del m
            m = CatBoostRegressor(**{**CAT_PARAMS, 'random_seed': seed})
            m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
                  eval_set=(X_val[feats], y_val_t), early_stopping_rounds=100, verbose=0)
            oof_s['cat'][val_idx] = np.maximum(target_inverse(m.predict(X_val[feats])), 0)
            test_s['cat'] += np.maximum(target_inverse(m.predict(X_te[feats])), 0) / 5
            del m
            f_lgb = mean_absolute_error(y_val, oof_s['lgb'][val_idx])
            f_xgb = mean_absolute_error(y_val, oof_s['xgb'][val_idx])
            f_cat = mean_absolute_error(y_val, oof_s['cat'][val_idx])
            print(f"  {label} F{fold+1} LGB={f_lgb:.4f} XGB={f_xgb:.4f} CAT={f_cat:.4f}")
            gc.collect()
        mae_l = mean_absolute_error(y_o, oof_s['lgb'][om])
        mae_x = mean_absolute_error(y_o, oof_s['xgb'][om])
        mae_c = mean_absolute_error(y_o, oof_s['cat'][om])
        inv = np.array([1/mae_l, 1/mae_x, 1/mae_c])
        w = inv / inv.sum()
        oof_e = w[0]*oof_s['lgb'][om] + w[1]*oof_s['xgb'][om] + w[2]*oof_s['cat'][om]
        test_e = np.maximum(w[0]*test_s['lgb'] + w[1]*test_s['xgb'] + w[2]*test_s['cat'], 0)
        mae_e = mean_absolute_error(y_o, oof_e)
        print(f"  {label} → Ens OOF: {mae_e:.4f} (LGB={mae_l:.4f} XGB={mae_x:.4f} CAT={mae_c:.4f})")
        return mae_e, test_e

# V4 1-seed 평가
mae_v4_1seed, test_v4_1seed = quick_eval_1seed(
    imp_train, test, features_v4, label="V4")

print(f"\n{'='*60}")
print(f"  Exp 84 V3 (10-seed): OOF 8.6392, LB 10.0737 (현 PB)")
print(f"  Exp 84 V3 (1-seed):  OOF 8.6397 (참고)")
print(f"  Exp 85 V4 (1-seed):  OOF {mae_v4_1seed:.4f}")
print(f"\n  V4 1-seed - V3 1-seed: {mae_v4_1seed - 8.6397:+.4f}")
print(f"  V4 1-seed - V3 10-seed: {mae_v4_1seed - 8.6392:+.4f}")
print(f"\n  주의: 1-seed → 10-seed 시 보통 OOF -0.005~0.01 개선됨")

# V3와 상관관계
corr_v4_v3, _ = pearsonr(result_v3['test_pred'], test_v4_1seed)
print(f"\n  V4 ↔ V3 상관관계: {corr_v4_v3:.4f}")
print(f"    (V3 ↔ V2: 0.9994)")

# 판단
v3_1seed_ref = 8.6397
diff = mae_v4_1seed - v3_1seed_ref

if diff < -0.005:
    print(f"\n  ✅ V4가 V3보다 명백히 좋음 ({-diff:.4f} 개선)")
    print(f"     → 10-seed 진행 권장, LB 새 PB 가능성")
elif diff < 0:
    print(f"\n  🟢 V4가 V3보다 미세 개선 ({-diff:.4f})")
    print(f"     → 10-seed에서 확장 가능성. 진행 권장")
elif diff < 0.005:
    print(f"\n  🟡 차이 미미 ({diff:+.4f})")
    print(f"     → 10-seed 진행해서 확인. fold 운빨 가능성")
else:
    print(f"\n  🔴 V4가 V3보다 나쁨 ({diff:+.4f})")
    print(f"     → percentile 확장 효과 없음. ablation 필요")
    print(f"     → 그래도 10-seed 시도 가능")


[Exp 85 Step 3] 1-seed 비교: V3(8.6392) vs V4


Default metric period is 5 because MAE is/are not implemented for GPU


  V4 F1 LGB=9.1861 XGB=9.2216 CAT=9.2737


Default metric period is 5 because MAE is/are not implemented for GPU


  V4 F2 LGB=8.0002 XGB=8.0114 CAT=8.0325


...

  V4 F3 LGB=6.4586 XGB=6.4985 CAT=6.4921


Default metric period is 5 because MAE is/are not implemented for GPU


  V4 F4 LGB=6.2898 XGB=6.3242 CAT=6.2769


Default metric period is 5 because MAE is/are not implemented for GPU


  V4 F5 LGB=7.6365 XGB=7.6495 CAT=7.6580
  V4 → Ens OOF: 8.6164 (LGB=8.6424 XGB=8.6451 CAT=8.6868)

  Exp 84 V3 (10-seed): OOF 8.6392, LB 10.0737 (현 PB)
  Exp 84 V3 (1-seed):  OOF 8.6397 (참고)
  Exp 85 V4 (1-seed):  OOF 8.6164

  V4 1-seed - V3 1-seed: -0.0233
  V4 1-seed - V3 10-seed: -0.0228

  주의: 1-seed → 10-seed 시 보통 OOF -0.005~0.01 개선됨

  V4 ↔ V3 상관관계: 0.9990
    (V3 ↔ V2: 0.9994)

  ✅ V4가 V3보다 명백히 좋음 (0.0233 개선)
     → 10-seed 진행 권장, LB 새 PB 가능성


In [ ]:
# Cell 4: V4 10-seed 학습 + 제출 파일

print(f"\n{'='*60}")
print(f"[Exp 85 Step 4] V4 10-seed 학습")
print(f"  피처: {len(features_v4)}개 + TE 4개")
print(f"  Train: {len(imp_train)}행 (Exp 79 R2 PL — 누수 없음)")
print("=" * 60)

result_v4 = train_gbdt_round(
    imp_train, test, features_v4, seeds, round_name="V4")

print(f"\n[Exp 85 V4 학습 완료]")
print(f"  V4 OOF MAE: {result_v4['oof_mae']:.4f}")
print(f"  V3 OOF MAE: 8.6392 (LB 10.0737)")
print(f"  V2 OOF MAE: 8.6438 (LB 10.0813)")
print(f"  V4 - V3 OOF: {result_v4['oof_mae'] - 8.6392:+.4f}")

# ── 제출 파일 생성 ──
print(f"\n[제출 파일 생성]")

pred_v4 = result_v4['test_pred']
pred_v3 = result_v3['test_pred']
pred_v2 = result_v2['test_pred']

# V4 단독
fname = 'submission_85_v4.csv'
pd.DataFrame({'ID': test['ID'], TARGET: pred_v4}).to_csv(
    os.path.join(DATA_DIR, fname), index=False)
print(f"  ✅ {fname} — V4 단독 (Percentile 확장)")

# V4 + V3 블렌딩
for alpha in [0.5, 0.7]:
    blend = alpha * pred_v4 + (1 - alpha) * pred_v3
    blend = np.maximum(blend, 0)
    fname = f'submission_85_v4_v3b{int(alpha*10)}.csv'
    pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(
        os.path.join(DATA_DIR, fname), index=False)
    print(f"  ✅ {fname} — V4 {int(alpha*100)}% + V3 {int((1-alpha)*100)}%")

# V4 + 75_g5 (TabNet 미세 다양성)
v75_path = os.path.join(DATA_DIR, 'submission_75_g5.csv')
if os.path.exists(v75_path):
    pred_75 = pd.read_csv(v75_path).set_index('ID').loc[test['ID'].values][TARGET].values
    blend = 0.9 * pred_v4 + 0.1 * pred_75
    blend = np.maximum(blend, 0)
    fname = 'submission_85_v4_75b1.csv'
    pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(
        os.path.join(DATA_DIR, fname), index=False)
    print(f"  ✅ {fname} — V4 90% + 75_g5 10%")

# ── 상관관계 분석 ──
corr_v4_v3_full, _ = pearsonr(pred_v4, pred_v3)
corr_v4_v2_full, _ = pearsonr(pred_v4, pred_v2)
print(f"\n  [상관관계 (10-seed)]")
print(f"    V4 ↔ V3: {corr_v4_v3_full:.4f}")
print(f"    V4 ↔ V2: {corr_v4_v2_full:.4f}")

# ── 모델별 ──
print(f"\n  [모델별 가중치]")
print(f"    LGB weight: {result_v4['weights'][0]:.4f}")
print(f"    XGB weight: {result_v4['weights'][1]:.4f}")
print(f"    CAT weight: {result_v4['weights'][2]:.4f}")

# ── 제출 우선순위 ──
print(f"\n  {'─'*55}")
print(f"  📋 제출 우선순위:")
print(f"  {'─'*55}")

if result_v4['oof_mae'] < 8.6392 - 0.003:
    # V4가 V3보다 명백히 좋음
    print(f"  1순위: submission_85_v4.csv")
    print(f"         (V4 OOF가 V3 대비 {8.6392 - result_v4['oof_mae']:.4f} 개선)")
    print(f"         예상 LB: 10.06~10.07")
    print(f"  2순위: submission_85_v4_v3b7.csv")
    print(f"         (V4 70% + V3 30% — 안전 보험)")
    print(f"  3순위: submission_85_v4_75b1.csv")
    print(f"         (V4 + TabNet 미세 다양성)")
elif result_v4['oof_mae'] < 8.6392 + 0.003:
    # 비슷
    print(f"  1순위: submission_85_v4.csv")
    print(f"         (OOF 비슷, percentile 확장 효과 LB에서 확인)")
    print(f"  2순위: submission_85_v4_v3b5.csv")
    print(f"         (50/50 블렌딩 — 가장 안전)")
    print(f"  3순위: submission_85_v4_v3b7.csv")
    print(f"         (V4 70% — 약한 V4 우세)")
else:
    # V4가 V3보다 나쁨
    print(f"  ⚠️ V4 OOF가 V3(8.6392)보다 나쁨")
    print(f"  1순위: submission_85_v4_v3b5.csv (50/50 안전)")
    print(f"  2순위: submission_85_v4.csv (V3↔V4 차이 LB 검증)")
    print(f"  → 다음 실험: V4 신규 그룹별 ablation 필요")

print(f"\n{'='*60}")
print(f"  Exp 85 V4 완료")
print(f"{'='*60}")


# Experiment 86: V5 피처 — Multi-dimensional Percentile 확장

**기반**: Exp 85 V4 (LB **10.0480** — 현재 PB), 219 피처

**목표**: LB 10.0480 → 10.03 진입

## V4 결과 검증 — Percentile Rank 가설 입증

V2 importance 분석에서 발견한 가설:
> "Scenario percentile rank가 GBDT가 학습하기 어려운 신호 — 3개로도 importance 1위. 35+개로 확장하면 큰 효과 기대."

**검증 완료**:
- V2 (3개 pctrank): LB 10.0813
- V4 (38개 pctrank): LB 10.0480
- Single-step LB **−0.0257** (전이율 0.89, OOF↔LB 정직 1:1)

V5는 이 검증된 신호를 **다차원으로 확장**합니다.

## V5 신규 피처 5그룹

| 그룹 | 내용 | 개수 | 가설 |
|------|------|------|------|
| **Q** | Layout 내 percentile (cross-scenario) | 8개 | 다른 시나리오 대비 위치 |
| **R** | Rolling 60 percentile | 10개 | 장기(60분) 상대 위치 |
| **S** | Percentile rank velocity | 8개 | rank가 오르내리는 추세 |
| **T** | Layout × percentile 교호작용 | 6개 | 구조 × 상대 위치 |
| **U** | Cumulative scenario rank | 6개 | 시작부터 지금까지 위치 |
| **합계** | | **38개** | V4와 다른 차원 |

### 가설 1: Layout 내 percentile (Q그룹)
V4까지는 scenario 내 위치만. **layout(여러 시나리오 묶음) 내 위치**는 새 차원.
"이 layout에서 평소보다 혼잡한가?"라는 신호.

### 가설 2: Rolling 60 percentile (R그룹)
V4의 rolling은 10, 30분. **60분** 추가로 시간 스케일 1차원 더.

### 가설 3: Rank velocity (S그룹) ⭐ 핵심 시도
Percentile rank의 변화량 — "rank가 빠르게 오르고 있는가?"
혼잡도 자체가 절대값 평균 수준이라도, rank가 급상승 중이면 미래 신호 강함.

### 가설 4: Layout × percentile (T그룹)
floor_area 큰 layout에서 percentile 90%인 것과 작은 layout에서 90%는 다른 의미.
구조와 상대 위치의 교호작용.

### 가설 5: Cumulative scenario rank (U그룹)
"전체 시나리오 진행 동안 지금이 가장 높은가?" — V4 since_max와 다른 형태의 사건 추적.

## 변경점 (V4 → V5)

| 항목 | V4 | V5 |
|------|-----|-----|
| 피처 수 | 219 | 목표 ~190 |
| 제거 | (V3 하위) | V4 importance 하위 80~100개 |
| 신규 | 45개 (M,N,O,P) | 38개 (Q,R,S,T,U) |
| Percentile 다양성 | scenario, roll10, roll30 | + layout, roll60, velocity, cumulative |

## 사전조건

Exp 85 V4 학습 완료 + 메모리에 다음:
- `result_v4`, `features_v4` (219개)
- `train_combined_r2` (Exp 79 R2 PL)
- `result_v3`, `result_v2`, `result_r2`, 학습 함수

## 실행 시간

- V4 importance 재분석: ~10분
- V5 피처 생성: ~10분
- 1-seed 비교: ~30분
- 10-seed 학습: ~2시간
- **합계: ~3시간**


In [ ]:
# Cell 1: V4 importance 재분석 → avg_rank_v4 생성

print("=" * 60)
print("[Exp 86 Step 1] V4 importance 재분석")
print("=" * 60)

# ── 사전조건 체크 ──
assert 'features_v4' in dir(), "features_v4 없음 — Exp 85 Cell 2 실행하세요"
assert 'result_v4' in dir(), "result_v4 없음 — Exp 85 학습 결과 필요"

imp_train = train_combined_r2 if 'train_combined_r2' in dir() else train_combined
print(f"  분석 대상: {len(imp_train)}행, {len(features_v4)}개 피처")

imp_seed = 42
imp_fold_target = 4
print(f"  학습: seed={imp_seed}, fold=F{imp_fold_target+1} only")

imp_lgb_v4 = {}
imp_xgb_v4 = {}
imp_cat_v4 = {}

for fold, (tr_idx, val_idx) in enumerate(
        gkf.split(imp_train, imp_train[TARGET],
                  groups=imp_train['layout_id'])):
    if fold != imp_fold_target:
        continue

    X_tr = imp_train.iloc[tr_idx].copy()
    y_tr = imp_train.iloc[tr_idx][TARGET].values
    sw   = make_sample_weight_v2(imp_train.iloc[tr_idx])
    X_val = imp_train.iloc[val_idx].copy()
    y_val = imp_train.iloc[val_idx][TARGET].values

    X_tr, X_val, te_cols = apply_smoothed_te(X_tr, X_val, TARGET, k=30)
    X_tr.fillna(0, inplace=True); X_val.fillna(0, inplace=True)

    feats = features_v4 + te_cols
    y_tr_t  = target_forward(y_tr); y_val_t = target_forward(y_val)

    print("  LGB 학습...")
    m = LGBMRegressor(**{**LGB_PARAMS, 'random_state': imp_seed})
    m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
          eval_set=[(X_val[feats], y_val_t)],
          callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
    for f, imp in zip(feats, m.feature_importances_):
        imp_lgb_v4[f] = imp
    del m; gc.collect()

    print("  XGB 학습...")
    m = xgb.XGBRegressor(**{**XGB_PARAMS, 'random_state': imp_seed})
    m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
          eval_set=[(X_val[feats], y_val_t)], verbose=False)
    for f, imp in zip(feats, m.feature_importances_):
        imp_xgb_v4[f] = imp
    del m; gc.collect()

    print("  CAT 학습...")
    m = CatBoostRegressor(**{**CAT_PARAMS, 'random_seed': imp_seed})
    m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
          eval_set=(X_val[feats], y_val_t),
          early_stopping_rounds=100, verbose=0)
    for f, imp in zip(feats, m.get_feature_importance()):
        imp_cat_v4[f] = imp
    del m; gc.collect()
    break

# ── DataFrame 통합 ──
imp_v4_df = pd.DataFrame({
    'feature': features_v4,
    'lgb_imp': [imp_lgb_v4.get(f, 0) for f in features_v4],
    'xgb_imp': [imp_xgb_v4.get(f, 0) for f in features_v4],
    'cat_imp': [imp_cat_v4.get(f, 0) for f in features_v4],
})

imp_v4_df['lgb_rank'] = imp_v4_df['lgb_imp'].rank(ascending=False, method='min')
imp_v4_df['xgb_rank'] = imp_v4_df['xgb_imp'].rank(ascending=False, method='min')
imp_v4_df['cat_rank'] = imp_v4_df['cat_imp'].rank(ascending=False, method='min')
imp_v4_df['avg_rank'] = imp_v4_df[['lgb_rank', 'xgb_rank', 'cat_rank']].mean(axis=1)
imp_v4_df = imp_v4_df.sort_values('avg_rank').reset_index(drop=True)

avg_rank_v4 = pd.Series(
    imp_v4_df['avg_rank'].values,
    index=imp_v4_df['feature'].values
).sort_values()

print(f"\n  분석 완료: {len(avg_rank_v4)}개 피처")

# ── V4 신규 피처 분석 (V4 검증 — percentile 효과 입증 핵심) ──
v4_new_feats = [f for f in features_v4 if f.startswith('v4_')]
print(f"\n  [V4 신규 피처 {len(v4_new_feats)}개의 rank 분포]")
new_ranks_v4 = avg_rank_v4[v4_new_feats]
print(f"    상위 10%: {(new_ranks_v4 <= len(features_v4)*0.10).sum()}개")
print(f"    상위 25%: {(new_ranks_v4 <= len(features_v4)*0.25).sum()}개")
print(f"    중위 25~75%: {((new_ranks_v4 > len(features_v4)*0.25) & (new_ranks_v4 <= len(features_v4)*0.75)).sum()}개")
print(f"    하위 25%: {(new_ranks_v4 > len(features_v4)*0.75).sum()}개")

# 그룹별 분석
print(f"\n  [V4 그룹별 평균 rank — 어느 그룹이 효과적이었나]")
groups = {
    'M (scen_pct)':  [f for f in v4_new_feats if 'scen_pct' in f],
    'N (roll30_pct)': [f for f in v4_new_feats if 'roll30_pct' in f],
    'O (roll10_pct)': [f for f in v4_new_feats if 'roll10_pct' in f],
    'P (since_max/min)': [f for f in v4_new_feats if 'since_max' in f or 'since_min' in f],
}
for gname, gfeats in groups.items():
    if gfeats:
        ranks = avg_rank_v4[gfeats]
        print(f"    {gname:25s}: {len(gfeats)}개, 평균 rank={ranks.mean():.1f}, 상위25%진입={((ranks <= len(features_v4)*0.25).sum())}개")

# 상위/하위
print(f"\n  [V4 신규 피처 상위 10]")
for i, (feat, r) in enumerate(avg_rank_v4[v4_new_feats].sort_values().head(10).items(), 1):
    print(f"    {i:2d}. {feat:55s}  rank={r:.1f}")

print(f"\n  [V4 신규 피처 하위 10 — V5에서 제거 후보]")
for i, (feat, r) in enumerate(avg_rank_v4[v4_new_feats].sort_values().tail(10).items(), 1):
    print(f"    {i:2d}. {feat:55s}  rank={r:.1f}")


In [ ]:
# Cell 2: V5 피처 정의 — Multi-dimensional Percentile 확장

print("=" * 60)
print("[Exp 86 Step 2] V5 피처 정의")
print("=" * 60)

# ── ① V4 하위 제거 ──
N_REMOVE_V5 = 80  # V5는 V4보다 보수적으로 (이미 잘 정제됨)
remove_v5 = set(avg_rank_v4.tail(N_REMOVE_V5).index)
remove_v5 -= {'layout_mean', 'layout_std', 'layout_median', 'layout_count'}

features_v5_slim = [f for f in features_v4 if f not in remove_v5]
print(f"  V4 피처: {len(features_v4)}개")
print(f"  제거: {len(remove_v5)}개 (V4 하위 {N_REMOVE_V5}개)")
print(f"  남은 피처: {len(features_v5_slim)}개")

# ── ② V5 신규 피처 5그룹 ──
print(f"\n[V5 신규 피처 생성]")
v5_new_feats = []

def col_exists(df, col):
    return col in df.columns

# --- Q. Layout 내 percentile (cross-scenario) ---
# 같은 layout의 모든 시나리오를 통틀어 본 위치
print(f"\n  Q. Layout 내 percentile (cross-scenario)")
LAYOUT_PCT_COLS = [
    'congestion_score', 'order_inflow_15m', 'pack_utilization',
    'robot_active', 'max_zone_density', 'pack_station_pressure',
    'loading_dock_util', 'fault_count_15m',
]

n_q = 0
for col in LAYOUT_PCT_COLS:
    if not col_exists(train_combined, col):
        continue
    fname = f'v5_{col}_layout_pct'
    for df in [train_combined, test]:
        df[fname] = df.groupby('layout_id')[col].rank(pct=True)
    v5_new_feats.append(fname)
    n_q += 1
print(f"     생성: {n_q}개")

# --- R. Rolling 60 percentile rank ---
# V4: 10, 30. V5: 60 추가 (장기 시간 스케일)
print(f"\n  R. Rolling 60 percentile rank")
ROLL60_COLS = [
    'congestion_score', 'order_inflow_15m', 'pack_utilization',
    'robot_active', 'max_zone_density', 'battery_mean',
    'pack_station_pressure', 'loading_dock_util',
    'fault_count_15m', 'blocked_path_15m',
]

n_r = 0
for col in ROLL60_COLS:
    if not col_exists(train_combined, col):
        continue
    fname = f'v5_{col}_roll60_pct'
    for df in [train_combined, test]:
        shifted = df.groupby('scenario_id')[col].shift(1)
        df[fname] = shifted.groupby(df['scenario_id']).transform(
            lambda x: x.rolling(60, min_periods=10).rank(pct=True)
        )
    v5_new_feats.append(fname)
    n_r += 1
print(f"     생성: {n_r}개")

# --- S. Percentile rank velocity (변화율) ---
# V4의 scen_pct 변화량 — rank가 오르고 있나
print(f"\n  S. Percentile rank velocity")
RANK_VELOCITY_COLS = [
    # V4에서 만들어진 scen_pct들 (가장 강력한 신호)
    'congestion_score_scen_pctrank',  # V2 신규
    'pack_utilization_scen_pctrank',  # V2 신규
    'order_inflow_15m_scen_pctrank',  # V2 신규
    'v4_robot_active_scen_pct',
    'v4_max_zone_density_scen_pct',
    'v4_pack_station_pressure_scen_pct',
    'v4_loading_dock_util_scen_pct',
    'v4_fault_count_15m_scen_pct',
]

n_s = 0
for col in RANK_VELOCITY_COLS:
    if not col_exists(train_combined, col):
        continue
    fname = f'v5_{col[:30]}_velocity'  # 이름 길어지지 않게
    for df in [train_combined, test]:
        df[fname] = df.groupby('scenario_id')[col].diff(1)
    v5_new_feats.append(fname)
    n_s += 1
print(f"     생성: {n_s}개")

# --- T. Layout × percentile 교호작용 ---
# 구조 컬럼과 percentile 결합 — 같은 percentile도 layout 크기에 따라 다른 의미
print(f"\n  T. Layout × percentile 교호작용")
LAYOUT_STRUCT_COLS = ['floor_area_sqm', 'pack_station_count', 'charger_count']
PCT_FOR_INTERACT = [
    'congestion_score_scen_pctrank',
    'pack_utilization_scen_pctrank',
]

n_t = 0
for layout_col in LAYOUT_STRUCT_COLS:
    if not col_exists(train_combined, layout_col):
        continue
    for pct_col in PCT_FOR_INTERACT:
        if not col_exists(train_combined, pct_col):
            continue
        fname = f'v5_{layout_col}_x_{pct_col[:25]}'
        for df in [train_combined, test]:
            df[fname] = df[layout_col] * df[pct_col]
        v5_new_feats.append(fname)
        n_t += 1
print(f"     생성: {n_t}개")

# --- U. Cumulative scenario rank ---
# 시나리오 시작부터 현재까지의 rank (점차 변하는 시간 위치)
print(f"\n  U. Cumulative scenario rank")
CUM_RANK_COLS = [
    'congestion_score', 'order_inflow_15m', 'pack_utilization',
    'robot_active', 'max_zone_density',
    'pack_station_pressure',
]

n_u = 0
for col in CUM_RANK_COLS:
    if not col_exists(train_combined, col):
        continue
    fname = f'v5_{col}_cum_pct'
    for df in [train_combined, test]:
        # expanding rank (현재까지의 누적 percentile)
        shifted = df.groupby('scenario_id')[col].shift(1)
        df[fname] = shifted.groupby(df['scenario_id']).transform(
            lambda x: x.expanding(min_periods=3).rank(pct=True)
        )
    v5_new_feats.append(fname)
    n_u += 1
print(f"     생성: {n_u}개")

# ── NaN 처리 ──
train_combined[v5_new_feats] = train_combined[v5_new_feats].fillna(0)
test[v5_new_feats] = test[v5_new_feats].fillna(0)

# ── train_combined_r2 동기화 ──
if 'train_combined_r2' in dir():
    if v5_new_feats[0] not in train_combined_r2.columns:
        print(f"\n  ⚠ train_combined_r2 재구성 중...")
        train_combined_r2_new, _ = build_pseudo_dataset(train, test, strategy='D')
        train_combined_r2 = train_combined_r2_new
        print(f"  ✓ 재구성 완료")
    else:
        print(f"\n  ✓ train_combined_r2 새 컬럼 자동 보유")

# ── 최종 V5 피처 세트 ──
features_v5 = features_v5_slim + v5_new_feats
print(f"\n  V5 신규 피처 합계: {len(v5_new_feats)}개")
print(f"  최종 V5 피처 세트: {len(features_v5)}개")
print(f"    = V4 {len(features_v4)} - 제거 {len(remove_v5)} + 신규 {len(v5_new_feats)}")
print(f"  V4 대비: {len(features_v5) - len(features_v4):+d}개")

# Percentile 카운트 추적
all_pct = sum(1 for f in features_v5 if 'pct' in f.lower() or 'pctrank' in f.lower())
print(f"\n  [Percentile 계열 피처 총 개수]")
print(f"    V2 시절: 3개 (pctrank)")
print(f"    V4 (M,N,O): 38개")
print(f"    V5 (V4 살아남음 + Q,R,S,T,U): {all_pct}개")


In [ ]:
# Cell 3: 1-seed 비교 — V4 vs V5

print("=" * 60)
print("[Exp 86 Step 3] 1-seed 비교: V4(8.6104) vs V5")
print("=" * 60)

imp_train = train_combined_r2 if 'train_combined_r2' in dir() else train_combined

if 'quick_eval_1seed' not in dir():
    print("  quick_eval_1seed 함수 정의...")
    def quick_eval_1seed(tc, test_df, feat_list, label="model", seed=42):
        om = ~tc['is_pseudo'].fillna(False).values
        y_o = tc[TARGET].values[om]
        oof_s = {k: np.zeros(len(tc)) for k in ['lgb','xgb','cat']}
        test_s = {k: np.zeros(len(test_df)) for k in ['lgb','xgb','cat']}
        for fold, (tr_idx, val_idx) in enumerate(
                gkf.split(tc, tc[TARGET], groups=tc['layout_id'])):
            X_tr = tc.iloc[tr_idx].copy()
            y_tr = tc.iloc[tr_idx][TARGET].values
            sw = make_sample_weight_v2(tc.iloc[tr_idx])
            X_val = tc.iloc[val_idx].copy()
            y_val = tc.iloc[val_idx][TARGET].values
            X_tr, X_val, te_cols = apply_smoothed_te(X_tr, X_val, TARGET, k=30)
            _, X_te, _ = apply_smoothed_te(X_tr, test_df.copy(), TARGET, k=30)
            X_tr.fillna(0, inplace=True); X_val.fillna(0, inplace=True); X_te.fillna(0, inplace=True)
            feats = feat_list + te_cols
            y_tr_t = target_forward(y_tr); y_val_t = target_forward(y_val)
            m = LGBMRegressor(**{**LGB_PARAMS, 'random_state': seed})
            m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
                  eval_set=[(X_val[feats], y_val_t)],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
            oof_s['lgb'][val_idx] = np.maximum(target_inverse(m.predict(X_val[feats])), 0)
            test_s['lgb'] += np.maximum(target_inverse(m.predict(X_te[feats])), 0) / 5
            del m
            m = xgb.XGBRegressor(**{**XGB_PARAMS, 'random_state': seed})
            m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
                  eval_set=[(X_val[feats], y_val_t)], verbose=False)
            oof_s['xgb'][val_idx] = np.maximum(target_inverse(m.predict(X_val[feats])), 0)
            test_s['xgb'] += np.maximum(target_inverse(m.predict(X_te[feats])), 0) / 5
            del m
            m = CatBoostRegressor(**{**CAT_PARAMS, 'random_seed': seed})
            m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
                  eval_set=(X_val[feats], y_val_t), early_stopping_rounds=100, verbose=0)
            oof_s['cat'][val_idx] = np.maximum(target_inverse(m.predict(X_val[feats])), 0)
            test_s['cat'] += np.maximum(target_inverse(m.predict(X_te[feats])), 0) / 5
            del m
            f_lgb = mean_absolute_error(y_val, oof_s['lgb'][val_idx])
            f_xgb = mean_absolute_error(y_val, oof_s['xgb'][val_idx])
            f_cat = mean_absolute_error(y_val, oof_s['cat'][val_idx])
            print(f"  {label} F{fold+1} LGB={f_lgb:.4f} XGB={f_xgb:.4f} CAT={f_cat:.4f}")
            gc.collect()
        mae_l = mean_absolute_error(y_o, oof_s['lgb'][om])
        mae_x = mean_absolute_error(y_o, oof_s['xgb'][om])
        mae_c = mean_absolute_error(y_o, oof_s['cat'][om])
        inv = np.array([1/mae_l, 1/mae_x, 1/mae_c])
        w = inv / inv.sum()
        oof_e = w[0]*oof_s['lgb'][om] + w[1]*oof_s['xgb'][om] + w[2]*oof_s['cat'][om]
        test_e = np.maximum(w[0]*test_s['lgb'] + w[1]*test_s['xgb'] + w[2]*test_s['cat'], 0)
        mae_e = mean_absolute_error(y_o, oof_e)
        print(f"  {label} → Ens OOF: {mae_e:.4f} (LGB={mae_l:.4f} XGB={mae_x:.4f} CAT={mae_c:.4f})")
        return mae_e, test_e

# V5 1-seed 평가
mae_v5_1seed, test_v5_1seed = quick_eval_1seed(
    imp_train, test, features_v5, label="V5")

print(f"\n{'='*60}")
print(f"  Exp 85 V4 (10-seed): OOF 8.6104, LB 10.0480 (현 PB)")
print(f"  Exp 85 V4 (1-seed):  OOF 8.6164 (참고)")
print(f"  Exp 86 V5 (1-seed):  OOF {mae_v5_1seed:.4f}")
print(f"\n  V5 1-seed - V4 1-seed: {mae_v5_1seed - 8.6164:+.4f}")
print(f"  V5 1-seed - V4 10-seed: {mae_v5_1seed - 8.6104:+.4f}")

# V4와 상관관계
corr_v5_v4, _ = pearsonr(result_v4['test_pred'], test_v5_1seed)
print(f"\n  V5 ↔ V4 상관관계: {corr_v5_v4:.4f}")
print(f"    (V4 ↔ V3: 0.9992, V3 ↔ V2: 0.9994 — 점차 감소 중)")

# 판단
v4_1seed_ref = 8.6164
diff = mae_v5_1seed - v4_1seed_ref

if diff < -0.01:
    print(f"\n  ✅ V5가 V4보다 매우 강력 ({-diff:.4f} 개선)")
    print(f"     → V4 도약(−0.0233) 패턴 재현 또는 더 큼")
    print(f"     → 10-seed 진행, 새 PB 매우 유력")
elif diff < -0.005:
    print(f"\n  ✅ V5가 V4보다 명백히 좋음 ({-diff:.4f} 개선)")
    print(f"     → 10-seed 진행, 새 PB 가능")
elif diff < 0:
    print(f"\n  🟢 V5가 V4보다 미세 개선 ({-diff:.4f})")
    print(f"     → 10-seed 진행해서 확인")
elif diff < 0.005:
    print(f"\n  🟡 차이 미미 ({diff:+.4f})")
    print(f"     → 10-seed 시도. V5 신규가 V4 ceiling에 가까운 신호")
else:
    print(f"\n  🔴 V5가 V4보다 나쁨 ({diff:+.4f})")
    print(f"     → percentile 추가 확장 효과 한계 도달")
    print(f"     → 신규 그룹 ablation 또는 다른 방향 (TabNet 등)")


In [ ]:
# Cell 4: V5 10-seed 학습 + 제출 파일

print(f"\n{'='*60}")
print(f"[Exp 86 Step 4] V5 10-seed 학습")
print(f"  피처: {len(features_v5)}개 + TE 4개")
print(f"  Train: {len(imp_train)}행 (Exp 79 R2 PL — 누수 없음)")
print("=" * 60)

result_v5 = train_gbdt_round(
    imp_train, test, features_v5, seeds, round_name="V5")

print(f"\n[Exp 86 V5 학습 완료]")
print(f"  V5 OOF MAE: {result_v5['oof_mae']:.4f}")
print(f"  V4 OOF MAE: 8.6104 (LB 10.0480)")
print(f"  V3 OOF MAE: 8.6392 (LB 10.0737)")
print(f"  V5 - V4 OOF: {result_v5['oof_mae'] - 8.6104:+.4f}")

# ── 제출 파일 ──
print(f"\n[제출 파일 생성]")

pred_v5 = result_v5['test_pred']
pred_v4 = result_v4['test_pred']
pred_v3 = result_v3['test_pred']

# V5 단독
fname = 'submission_86_v5.csv'
pd.DataFrame({'ID': test['ID'], TARGET: pred_v5}).to_csv(
    os.path.join(DATA_DIR, fname), index=False)
print(f"  ✅ {fname} — V5 단독 (Multi-dim Percentile 확장)")

# V5 + V4 블렌딩
for alpha in [0.5, 0.7]:
    blend = alpha * pred_v5 + (1 - alpha) * pred_v4
    blend = np.maximum(blend, 0)
    fname = f'submission_86_v5_v4b{int(alpha*10)}.csv'
    pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(
        os.path.join(DATA_DIR, fname), index=False)
    print(f"  ✅ {fname} — V5 {int(alpha*100)}% + V4 {int((1-alpha)*100)}%")

# V5 + 75_g5 (TabNet 미세 다양성)
v75_path = os.path.join(DATA_DIR, 'submission_75_g5.csv')
if os.path.exists(v75_path):
    pred_75 = pd.read_csv(v75_path).set_index('ID').loc[test['ID'].values][TARGET].values
    blend = 0.9 * pred_v5 + 0.1 * pred_75
    blend = np.maximum(blend, 0)
    fname = 'submission_86_v5_75b1.csv'
    pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(
        os.path.join(DATA_DIR, fname), index=False)
    print(f"  ✅ {fname} — V5 90% + 75_g5 10% (TabNet 다양성)")

# ── 상관관계 ──
corr_v5_v4_full, _ = pearsonr(pred_v5, pred_v4)
corr_v5_v3_full, _ = pearsonr(pred_v5, pred_v3)
print(f"\n  [상관관계 (10-seed)]")
print(f"    V5 ↔ V4: {corr_v5_v4_full:.4f}")
print(f"    V5 ↔ V3: {corr_v5_v3_full:.4f}")

# 모델별 가중치
print(f"\n  [모델별 가중치]")
print(f"    LGB weight: {result_v5['weights'][0]:.4f}")
print(f"    XGB weight: {result_v5['weights'][1]:.4f}")
print(f"    CAT weight: {result_v5['weights'][2]:.4f}")

# ── 제출 우선순위 ──
print(f"\n  {'─'*55}")
print(f"  📋 제출 우선순위:")
print(f"  {'─'*55}")

if result_v5['oof_mae'] < 8.6104 - 0.01:
    print(f"  1순위: submission_86_v5.csv")
    print(f"         (V5 OOF가 V4 대비 {8.6104 - result_v5['oof_mae']:.4f} 큰 폭 개선)")
    print(f"         예상 LB: 10.02~10.04")
    print(f"  2순위: submission_86_v5_v4b7.csv (V5 70% + V4 30% 보험)")
    print(f"  3순위: submission_86_v5_75b1.csv")
elif result_v5['oof_mae'] < 8.6104:
    print(f"  1순위: submission_86_v5.csv")
    print(f"         (V5 OOF가 V4 대비 {8.6104 - result_v5['oof_mae']:.4f} 개선)")
    print(f"  2순위: submission_86_v5_v4b7.csv (V5 70% + V4 30%)")
    print(f"  3순위: submission_86_v5_75b1.csv")
else:
    print(f"  ⚠️ V5 OOF가 V4(8.6104)보다 나쁨 ({result_v5['oof_mae'] - 8.6104:+.4f})")
    print(f"  1순위: submission_86_v5_v4b5.csv (50/50 안전)")
    print(f"  2순위: submission_86_v5.csv (V5 단독, 검증 가치)")
    print(f"  → 다음 실험: V5 그룹 ablation (Q,R,S,T,U 중 핵심 식별)")

print(f"\n{'='*60}")
print(f"  Exp 86 V5 완료")
print(f"{'='*60}")


# Experiment 87: V4.5 피처 — 정밀 정제 (V4 진단 결과 반영)

**기반**: Exp 85 V4 (LB **10.0480** — 현재 PB), 219 피처

**목표**: LB 10.0480 → 10.035~10.045 진입

## 동기 — V5 1-seed에서 V4 ceiling 발견

V5 1-seed 결과 (V4 1-seed 대비 +0.0019, 약함):
- Q(layout_pct), R(roll60_pct), S(velocity), T(interaction), U(cumulative) — 모두 V4 진단과 충돌
- **결론**: percentile은 다양한 차원이 아니라 **specific한 형태(scen_pct)가 핵심**

## V4 importance 진단 — 핵심 발견

V4 신규 피처 42개의 그룹별 분포:

| 그룹 | 개수 | 평균 rank | 상위 25% 진입 | 효과 |
|------|------|----------|--------------|------|
| **M (scen_pct)** | 12 | 141.6 | **3개** | ⭐ Effective |
| N (roll30_pct) | 12 | 193.9 | 0 | ❌ Noise |
| O (roll10_pct) | 8 | 206.6 | 0 | ❌ Noise |
| P (since_max/min) | 10 | 192.4 | 0 | ❌ Noise |

V4 효과의 본질은 **단 3개의 강력한 피처**:

```
v4_max_zone_density_scen_pct                  rank=24.0  ⭐
v4_pack_utilization_roll10_mean_scen_pct      rank=33.3  ⭐
v4_congestion_score_roll10_mean_scen_pct      rank=51.0  ⭐
```

**패턴 추출**: `(raw or roll10_mean) → scen_pct`가 강력.

## V4.5 전략

### 1. 제거: noise 식별된 V4 신규 (39개)

- N 그룹 12개 (roll30_pct) — 평균 rank 193.9
- O 그룹 8개 (roll10_pct) — 평균 rank 206.6
- P 그룹 10개 (since_max/min) — 평균 rank 192.4
- M 그룹 12개 중 하위 9개 (rank > 100) — 상위 3개만 유지
- avg_rank_v4 추가 하위 30~50개

### 2. 추가: M 패턴 확장 (effective한 형태로만)

- `roll10_mean → scen_pct`: 더 많은 컬럼에 적용 (V4 분석에서 rank 33, 51 입증)
- `roll5_mean → scen_pct`: 단기 신호
- `roll20_mean → scen_pct`: 중기 신호
- raw value scen_pct: V4의 max_zone_density_scen_pct(rank 24) 패턴 확장
- **N, O, P 패턴은 절대 추가 안 함**

## 기대 효과

```
V4 효과 분해:
  V4 신규 42개 → 효과 있는 3개만 → LB −0.026 도약

V4.5 가설:
  effective 패턴(rolling mean → scen_pct)을 8~12개로 확장
  → 같은 패턴의 효과 누적
  → noise 39개 제거로 추가 개선
  → 기대 LB: 10.035~10.045
```

## 사전조건

Exp 85 V4 학습 완료 + 메모리에:
- `result_v4`, `features_v4` (219개)
- `avg_rank_v4` (V4 importance 분석 결과)
- `train_combined_r2`, 학습 함수들

## 실행 시간

- V4.5 피처 정의: ~5분 (avg_rank_v4 활용)
- 1-seed 비교: ~30분
- 10-seed 학습: ~2시간
- **합계: ~2시간 30분**


In [ ]:
# Cell 1: V4.5 피처 정의 — V4 진단 직접 반영

print("=" * 60)
print("[Exp 87 Step 1] V4.5 피처 정의")
print("=" * 60)

# ── 사전조건 체크 ──
assert 'features_v4' in dir(), "features_v4 없음"
assert 'avg_rank_v4' in dir(), "avg_rank_v4 없음 — Exp 86 Cell 1 실행하세요"
assert 'result_v4' in dir(), "result_v4 없음"

# ── ① 제거 대상 식별 ──
print(f"\n[제거 대상 식별]")

# V4 신규 피처 분류
v4_new_feats = [f for f in features_v4 if f.startswith('v4_')]
m_group = [f for f in v4_new_feats if 'scen_pct' in f]
n_group = [f for f in v4_new_feats if 'roll30_pct' in f]
o_group = [f for f in v4_new_feats if 'roll10_pct' in f]
p_group = [f for f in v4_new_feats if 'since_max' in f or 'since_min' in f]

print(f"  V4 신규 그룹 분류:")
print(f"    M (scen_pct):     {len(m_group)}개")
print(f"    N (roll30_pct):   {len(n_group)}개  → 모두 제거")
print(f"    O (roll10_pct):   {len(o_group)}개  → 모두 제거")
print(f"    P (since_max/min):{len(p_group)}개  → 모두 제거")

# M 그룹에서 상위만 유지 — V4 진단 결과의 핵심 3개
m_top_threshold = 100  # rank 100 이하만 유지 (V4 진단에서 의미있는 cutoff)
m_top = [f for f in m_group if avg_rank_v4[f] <= m_top_threshold]
m_bottom = [f for f in m_group if avg_rank_v4[f] > m_top_threshold]
print(f"\n  M 그룹 정제 (rank ≤ {m_top_threshold}):")
print(f"    유지: {len(m_top)}개")
for f in sorted(m_top, key=lambda x: avg_rank_v4[x]):
    print(f"      ✓ {f:55s}  rank={avg_rank_v4[f]:.1f}")
print(f"    제거: {len(m_bottom)}개")

# 추가 제거 — V4 importance 하위
N_REMOVE_BY_RANK = 30  # 신규 외 V4 기존 피처 중 하위
existing_features = [f for f in features_v4 if not f.startswith('v4_')]
existing_ranks = avg_rank_v4[[f for f in existing_features if f in avg_rank_v4.index]]
remove_existing = set(existing_ranks.sort_values().tail(N_REMOVE_BY_RANK).index)

# 합집합
remove_v45 = (set(n_group) | set(o_group) | set(p_group) | 
              set(m_bottom) | remove_existing)

# TE 피처 보호
remove_v45 -= {'layout_mean', 'layout_std', 'layout_median', 'layout_count'}

features_v45_slim = [f for f in features_v4 if f not in remove_v45]
print(f"\n  총 제거: {len(remove_v45)}개")
print(f"    - N, O, P 그룹: {len(n_group) + len(o_group) + len(p_group)}개")
print(f"    - M 그룹 하위:  {len(m_bottom)}개")
print(f"    - 기존 하위:    {N_REMOVE_BY_RANK}개")
print(f"  남은 피처: {len(features_v45_slim)}개 (V4 219 - {len(remove_v45)})")

# ── ② V4.5 신규 피처 — effective 패턴 확장 ──
print(f"\n[V4.5 신규 피처 생성 — M 패턴 확장]")
v45_new_feats = []

def col_exists(df, col):
    return col in df.columns

# M 효과의 핵심 패턴: (raw or rolling mean) → scen_pct
# V4 분석의 상위 3개를 일반화

# 핵심 컬럼 — V4에서 effective scen_pct 입증된 컬럼 + 확장
PCT_TARGETS = [
    # 이미 V4에서 effective 입증된 (재생성 안 함, 이미 features_v45_slim에 있음)
    # 'max_zone_density',           — V4의 raw scen_pct (rank 24)
    # 'pack_utilization_roll10_mean', — V4의 (rank 33)
    # 'congestion_score_roll10_mean', — V4의 (rank 51)
    
    # V4.5에서 추가할 컬럼 (M 패턴 일반화)
    'order_inflow_15m',      # 핵심 운영 지표
    'robot_active',
    'pack_station_pressure',
    'fault_count_15m',
    'blocked_path_15m',
    'battery_mean',
    'loading_dock_util',
    'staging_area_util',
    'robot_idle',
    'robot_charging',
]

# 패턴 1: raw value → scen_pct (V4의 max_zone_density 패턴)
print(f"\n  V. Raw value → scen_pct (M 패턴 일반화)")
n_v_raw = 0
for col in PCT_TARGETS:
    if not col_exists(train_combined, col):
        continue
    fname = f'v45_{col}_scen_pct'
    # 이미 features_v45_slim에 같은 이름이 있는지 체크
    if fname in features_v45_slim:
        continue
    for df in [train_combined, test]:
        df[fname] = df.groupby('scenario_id')[col].rank(pct=True)
    v45_new_feats.append(fname)
    n_v_raw += 1
print(f"     생성: {n_v_raw}개")

# 패턴 2: rolling mean (5, 10, 20) → scen_pct (V4의 강력 패턴)
print(f"\n  W. Rolling mean → scen_pct (V4 rank 33, 51 패턴)")
ROLL_MEAN_TARGETS = [
    # V4에서 검증된 + 확장
    'order_inflow_15m', 'robot_active', 'pack_station_pressure',
    'max_zone_density',  # V4 raw로만 있고 rolling mean은 없을 수 있음
    'fault_count_15m', 'battery_mean',
]

n_w = 0
for col in ROLL_MEAN_TARGETS:
    for window in [5, 10, 20]:  # V4의 10에 5, 20 추가
        roll_col = f'{col}_roll{window}_mean'
        if not col_exists(train_combined, roll_col):
            continue
        fname = f'v45_{col}_roll{window}m_scen_pct'
        if fname in features_v45_slim:
            continue
        for df in [train_combined, test]:
            df[fname] = df.groupby('scenario_id')[roll_col].rank(pct=True)
        v45_new_feats.append(fname)
        n_w += 1
print(f"     생성: {n_w}개")

# 패턴 3: 핵심 vs_cummax/cummin → scen_pct (보조 시도)
# V4에 'max_zone_density_vs_cummax'가 있었지만 importance 낮았음.
# 그러나 vs_cummax 자체가 percentile-like → scen_pct 적용 시 어떨지
print(f"\n  X. vs_cummax/cummin → scen_pct (시도)")
VS_CUMMAX_TARGETS = ['congestion_score', 'order_inflow_15m', 'pack_utilization']
n_x = 0
for col in VS_CUMMAX_TARGETS:
    for cummax_type in ['vs_cummax', 'vs_cummin']:
        src_col = f'{col}_{cummax_type}'
        if not col_exists(train_combined, src_col):
            continue
        fname = f'v45_{col}_{cummax_type}_scen_pct'
        if fname in features_v45_slim:
            continue
        for df in [train_combined, test]:
            df[fname] = df.groupby('scenario_id')[src_col].rank(pct=True)
        v45_new_feats.append(fname)
        n_x += 1
print(f"     생성: {n_x}개")

# ── NaN 처리 ──
train_combined[v45_new_feats] = train_combined[v45_new_feats].fillna(0)
test[v45_new_feats] = test[v45_new_feats].fillna(0)

# ── train_combined_r2 동기화 ──
if 'train_combined_r2' in dir():
    if v45_new_feats and v45_new_feats[0] not in train_combined_r2.columns:
        print(f"\n  ⚠ train_combined_r2 재구성 중...")
        train_combined_r2_new, _ = build_pseudo_dataset(train, test, strategy='D')
        train_combined_r2 = train_combined_r2_new
        print(f"  ✓ 재구성 완료")
    else:
        print(f"\n  ✓ train_combined_r2 새 컬럼 자동 보유")

# ── 최종 V4.5 피처 세트 ──
features_v45 = features_v45_slim + v45_new_feats
print(f"\n  V4.5 신규 피처 합계: {len(v45_new_feats)}개")
print(f"  최종 V4.5 피처 세트: {len(features_v45)}개")
print(f"    = V4 {len(features_v4)} - 제거 {len(remove_v45)} + 신규 {len(v45_new_feats)}")
print(f"  V4 대비: {len(features_v45) - len(features_v4):+d}개")

# Percentile 계열 카운트
total_pct = sum(1 for f in features_v45 if 'scen_pct' in f or 'pctrank' in f)
print(f"\n  [scen_pct/pctrank 계열 총 개수]")
print(f"    V2 시절: 3개")
print(f"    V4: ~15개 (M 그룹 + V2)")
print(f"    V4.5: {total_pct}개")


In [ ]:
# Cell 2: 1-seed 비교 — V4 vs V4.5

print("=" * 60)
print("[Exp 87 Step 2] 1-seed 비교: V4(8.6104) vs V4.5")
print("=" * 60)

imp_train = train_combined_r2 if 'train_combined_r2' in dir() else train_combined

if 'quick_eval_1seed' not in dir():
    print("  quick_eval_1seed 함수 정의...")
    def quick_eval_1seed(tc, test_df, feat_list, label="model", seed=42):
        om = ~tc['is_pseudo'].fillna(False).values
        y_o = tc[TARGET].values[om]
        oof_s = {k: np.zeros(len(tc)) for k in ['lgb','xgb','cat']}
        test_s = {k: np.zeros(len(test_df)) for k in ['lgb','xgb','cat']}
        for fold, (tr_idx, val_idx) in enumerate(
                gkf.split(tc, tc[TARGET], groups=tc['layout_id'])):
            X_tr = tc.iloc[tr_idx].copy()
            y_tr = tc.iloc[tr_idx][TARGET].values
            sw = make_sample_weight_v2(tc.iloc[tr_idx])
            X_val = tc.iloc[val_idx].copy()
            y_val = tc.iloc[val_idx][TARGET].values
            X_tr, X_val, te_cols = apply_smoothed_te(X_tr, X_val, TARGET, k=30)
            _, X_te, _ = apply_smoothed_te(X_tr, test_df.copy(), TARGET, k=30)
            X_tr.fillna(0, inplace=True); X_val.fillna(0, inplace=True); X_te.fillna(0, inplace=True)
            feats = feat_list + te_cols
            y_tr_t = target_forward(y_tr); y_val_t = target_forward(y_val)
            m = LGBMRegressor(**{**LGB_PARAMS, 'random_state': seed})
            m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
                  eval_set=[(X_val[feats], y_val_t)],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
            oof_s['lgb'][val_idx] = np.maximum(target_inverse(m.predict(X_val[feats])), 0)
            test_s['lgb'] += np.maximum(target_inverse(m.predict(X_te[feats])), 0) / 5
            del m
            m = xgb.XGBRegressor(**{**XGB_PARAMS, 'random_state': seed})
            m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
                  eval_set=[(X_val[feats], y_val_t)], verbose=False)
            oof_s['xgb'][val_idx] = np.maximum(target_inverse(m.predict(X_val[feats])), 0)
            test_s['xgb'] += np.maximum(target_inverse(m.predict(X_te[feats])), 0) / 5
            del m
            m = CatBoostRegressor(**{**CAT_PARAMS, 'random_seed': seed})
            m.fit(X_tr[feats], y_tr_t, sample_weight=sw,
                  eval_set=(X_val[feats], y_val_t), early_stopping_rounds=100, verbose=0)
            oof_s['cat'][val_idx] = np.maximum(target_inverse(m.predict(X_val[feats])), 0)
            test_s['cat'] += np.maximum(target_inverse(m.predict(X_te[feats])), 0) / 5
            del m
            f_lgb = mean_absolute_error(y_val, oof_s['lgb'][val_idx])
            f_xgb = mean_absolute_error(y_val, oof_s['xgb'][val_idx])
            f_cat = mean_absolute_error(y_val, oof_s['cat'][val_idx])
            print(f"  {label} F{fold+1} LGB={f_lgb:.4f} XGB={f_xgb:.4f} CAT={f_cat:.4f}")
            gc.collect()
        mae_l = mean_absolute_error(y_o, oof_s['lgb'][om])
        mae_x = mean_absolute_error(y_o, oof_s['xgb'][om])
        mae_c = mean_absolute_error(y_o, oof_s['cat'][om])
        inv = np.array([1/mae_l, 1/mae_x, 1/mae_c])
        w = inv / inv.sum()
        oof_e = w[0]*oof_s['lgb'][om] + w[1]*oof_s['xgb'][om] + w[2]*oof_s['cat'][om]
        test_e = np.maximum(w[0]*test_s['lgb'] + w[1]*test_s['xgb'] + w[2]*test_s['cat'], 0)
        mae_e = mean_absolute_error(y_o, oof_e)
        print(f"  {label} → Ens OOF: {mae_e:.4f} (LGB={mae_l:.4f} XGB={mae_x:.4f} CAT={mae_c:.4f})")
        return mae_e, test_e

# V4.5 1-seed 평가
mae_v45_1seed, test_v45_1seed = quick_eval_1seed(
    imp_train, test, features_v45, label="V4.5")

print(f"\n{'='*60}")
print(f"  Exp 85 V4 (10-seed): OOF 8.6104, LB 10.0480 (현 PB)")
print(f"  Exp 85 V4 (1-seed):  OOF 8.6164")
print(f"  Exp 86 V5 (1-seed):  OOF 8.6183 (V4 대비 +0.0019, V4 ceiling 시사)")
print(f"  Exp 87 V4.5 (1-seed): OOF {mae_v45_1seed:.4f}")
print(f"\n  V4.5 1-seed - V4 1-seed: {mae_v45_1seed - 8.6164:+.4f}")
print(f"  V4.5 1-seed - V4 10-seed: {mae_v45_1seed - 8.6104:+.4f}")

# V4와 상관관계
corr_v45_v4, _ = pearsonr(result_v4['test_pred'], test_v45_1seed)
print(f"\n  V4.5 ↔ V4 상관관계: {corr_v45_v4:.4f}")
print(f"    (V4 ↔ V3: 0.9992, V5 ↔ V4: 0.9994)")

# 판단
v4_1seed_ref = 8.6164
diff = mae_v45_1seed - v4_1seed_ref

if diff < -0.01:
    print(f"\n  ✅ V4.5가 V4보다 매우 강력 ({-diff:.4f} 개선)")
    print(f"     → effective 패턴 확장 + noise 제거 효과 확인")
    print(f"     → 10-seed 진행, 새 PB 매우 유력")
elif diff < -0.005:
    print(f"\n  ✅ V4.5가 V4보다 명백히 좋음 ({-diff:.4f} 개선)")
    print(f"     → 10-seed 진행, 새 PB 가능")
elif diff < 0:
    print(f"\n  🟢 V4.5가 V4보다 미세 개선 ({-diff:.4f})")
    print(f"     → 10-seed 진행")
elif diff < 0.003:
    print(f"\n  🟡 차이 미미 ({diff:+.4f})")
    print(f"     → 10-seed 시도. V4 ceiling 신호 더 강해짐")
    print(f"     → ROI 미묘. TabNet 전환 검토")
else:
    print(f"\n  🔴 V4.5가 V4보다 나쁨 ({diff:+.4f})")
    print(f"     → 정제 효과 없음. V4가 정말 ceiling")
    print(f"     → 다음 방향: TabNet (구조적 다양성)")
    print(f"     → 또는 PL 기반 변경 (Exp 79 R2 → V4 PL 부분 활용)")


In [ ]:
# Cell 3: V4.5 10-seed 학습 + 제출 파일

print(f"\n{'='*60}")
print(f"[Exp 87 Step 3] V4.5 10-seed 학습")
print(f"  피처: {len(features_v45)}개 + TE 4개")
print(f"  Train: {len(imp_train)}행 (Exp 79 R2 PL — 누수 없음)")
print("=" * 60)

result_v45 = train_gbdt_round(
    imp_train, test, features_v45, seeds, round_name="V4.5")

print(f"\n[Exp 87 V4.5 학습 완료]")
print(f"  V4.5 OOF MAE: {result_v45['oof_mae']:.4f}")
print(f"  V4 OOF MAE: 8.6104 (LB 10.0480)")
print(f"  V4.5 - V4 OOF: {result_v45['oof_mae'] - 8.6104:+.4f}")

# ── 제출 파일 ──
print(f"\n[제출 파일 생성]")

pred_v45 = result_v45['test_pred']
pred_v4 = result_v4['test_pred']
pred_v3 = result_v3['test_pred']

# V4.5 단독
fname = 'submission_87_v45.csv'
pd.DataFrame({'ID': test['ID'], TARGET: pred_v45}).to_csv(
    os.path.join(DATA_DIR, fname), index=False)
print(f"  ✅ {fname} — V4.5 단독 (정밀 정제)")

# V4.5 + V4 블렌딩
for alpha in [0.5, 0.7]:
    blend = alpha * pred_v45 + (1 - alpha) * pred_v4
    blend = np.maximum(blend, 0)
    fname = f'submission_87_v45_v4b{int(alpha*10)}.csv'
    pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(
        os.path.join(DATA_DIR, fname), index=False)
    print(f"  ✅ {fname} — V4.5 {int(alpha*100)}% + V4 {int((1-alpha)*100)}%")

# V4.5 + 75_g5 (TabNet 미세 다양성)
v75_path = os.path.join(DATA_DIR, 'submission_75_g5.csv')
if os.path.exists(v75_path):
    pred_75 = pd.read_csv(v75_path).set_index('ID').loc[test['ID'].values][TARGET].values
    blend = 0.9 * pred_v45 + 0.1 * pred_75
    blend = np.maximum(blend, 0)
    fname = 'submission_87_v45_75b1.csv'
    pd.DataFrame({'ID': test['ID'], TARGET: blend}).to_csv(
        os.path.join(DATA_DIR, fname), index=False)
    print(f"  ✅ {fname} — V4.5 90% + 75_g5 10% (TabNet 다양성)")

# ── 상관관계 ──
corr_v45_v4_full, _ = pearsonr(pred_v45, pred_v4)
print(f"\n  [상관관계 (10-seed)]")
print(f"    V4.5 ↔ V4: {corr_v45_v4_full:.4f}")

# 모델별 가중치
print(f"\n  [모델별 가중치]")
print(f"    LGB weight: {result_v45['weights'][0]:.4f}")
print(f"    XGB weight: {result_v45['weights'][1]:.4f}")
print(f"    CAT weight: {result_v45['weights'][2]:.4f}")

# ── 제출 우선순위 ──
print(f"\n  {'─'*55}")
print(f"  📋 제출 우선순위:")
print(f"  {'─'*55}")

if result_v45['oof_mae'] < 8.6104 - 0.005:
    print(f"  1순위: submission_87_v45.csv")
    print(f"         (V4.5 OOF가 V4 대비 {8.6104 - result_v45['oof_mae']:.4f} 큰 개선)")
    print(f"         예상 LB: 10.035~10.045")
    print(f"  2순위: submission_87_v45_v4b7.csv (V4.5 70% + V4 30%)")
    print(f"  3순위: submission_87_v45_75b1.csv (V4.5 + TabNet)")
elif result_v45['oof_mae'] < 8.6104:
    print(f"  1순위: submission_87_v45.csv")
    print(f"         (V4.5 OOF가 V4 대비 {8.6104 - result_v45['oof_mae']:.4f} 개선)")
    print(f"  2순위: submission_87_v45_v4b7.csv")
    print(f"  3순위: submission_87_v45_75b1.csv")
else:
    print(f"  ⚠️ V4.5 OOF가 V4(8.6104)보다 나쁨 ({result_v45['oof_mae'] - 8.6104:+.4f})")
    print(f"  1순위: submission_87_v45_v4b5.csv (50/50 안전)")
    print(f"  2순위: submission_87_v45.csv (검증)")
    print(f"  → V4가 진정한 GBDT ceiling 신호")
    print(f"  → 다음: TabNet (구조적 다양성), 또는 PL refinement 변형")

print(f"\n{'='*60}")
print(f"  Exp 87 V4.5 완료")
print(f"{'='*60}")
